# BERT

In [1]:
!curl -L -o ~/invoices-ocr.zip https://www.kaggle.com/api/v1/datasets/download/osamahosamabdellatif/high-quality-invoice-images-for-ocr
!unzip ~/invoices-ocr.zip -d /content/invoices

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1165M  100 1165M    0     0  16.4M      0  0:01:10  0:01:10 --:--:-- 16.8M
Archive:  /root/invoices-ocr.zip
  inflating: /content/invoices/batch_1/batch_1/batch1_1.csv  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0001.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0002.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0003.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0004.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0005.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0006.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0007.jpg  
  inflating: /content/invoices/batch_1/batch_1/batch1_1/batch1-0008.jpg

In [2]:
!pip install -q \
    transformers \
    datasets \
    accelerate \
    sentencepiece \
    scikit-learn
!pip install -q --use-pep517 seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
import json
import random
import re

import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from pathlib import Path
import unicodedata

from pydantic_core import core_schema

from datasets import Dataset, DatasetDict
from collections.abc import Callable
from collections import defaultdict
from sklearn.model_selection import train_test_split
from seqeval.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from pydantic import BaseModel
from enum import StrEnum

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [7]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA: True
GPU: NVIDIA A100-SXM4-40GB


In [8]:
DATA_PATH = Path("/content/drive/MyDrive/ocr/annotated_invoices.parquet")

In [9]:
columns = [
    "path",
    "file_name",
    "subdir",
    "source_group",
    "ocred_text",
    "json_data",
]

In [10]:
df = pd.read_parquet(DATA_PATH)
df.head()

,file_name,json_data,ocred_text,subdir,source_group,csv_path,path,target,top_level_keys,target_text
0,batch1-0494.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Clark...",Invoice no: 84652373 Date of issue: 02/23/2021...,batch1_1,batch_1/batch_1,/content/invoices/batch_1/batch_1/batch1_1.csv,/content/invoices/batch_1/batch_1/batch1_1/bat...,{'invoice': {'client_address': '77477 Troy Cli...,"[invoice, items, payment_instructions, subtotal]","{""invoice"":{""client_address"":""77477 Troy Cliff..."
1,batch1-0489.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Willi...",Invoice no: 37451664 Date of issue: 06/11/2020...,batch1_1,batch_1/batch_1,/content/invoices/batch_1/batch_1/batch1_1.csv,/content/invoices/batch_1/batch_1/batch1_1/bat...,{'invoice': {'client_address': '588 Morales Ra...,"[invoice, items, payment_instructions, subtotal]","{""invoice"":{""client_address"":""588 Morales Rapi..."
2,batch1-0499.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Heste...",Invoice no: 40108666 Date of issue: 02/07/2020...,batch1_1,batch_1/batch_1,/content/invoices/batch_1/batch_1/batch1_1.csv,/content/invoices/batch_1/batch_1/batch1_1/bat...,{'invoice': {'client_address': '4000 Tonya Rap...,"[invoice, items, payment_instructions, subtotal]","{""invoice"":{""client_address"":""4000 Tonya Rapid..."
3,batch1-0497.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Olson...",Invoice no: 73285932 Date of issue: 07/25/2017...,batch1_1,batch_1/batch_1,/content/invoices/batch_1/batch_1/batch1_1.csv,/content/invoices/batch_1/batch_1/batch1_1/bat...,{'invoice': {'client_address': '80943 Beck Div...,"[invoice, items, payment_instructions, subtotal]","{""invoice"":{""client_address"":""80943 Beck Divid..."
4,batch1-0081.jpg,"\n{\n ""invoice"": {\n ""client_name"": ""Wilso...",Invoice no: 15288019 Date of issue: 09/07/2014...,batch1_1,batch_1/batch_1,/content/invoices/batch_1/batch_1/batch1_1.csv,/content/invoices/batch_1/batch_1/batch1_1/bat...,{'invoice': {'client_address': '85455 Carter E...,"[invoice, items, payment_instructions, subtotal]","{""invoice"":{""client_address"":""85455 Carter Est..."


In [11]:
PHASE_1_FIELDS = [
    "invoice.invoice_number",
    "invoice.invoice_date",
    "invoice.due_date",
    "invoice.seller_name",
    "invoice.seller_address",
    "invoice.client_name",
    "invoice.client_address",
    "subtotal.tax",
    "subtotal.discount",
    "subtotal.total",
    "payment_instructions.bank_name",
    "payment_instructions.account_number",
    "payment_instructions.payment_method",
]

ENTITY_TYPES = [
    "INVOICE_NUMBER",
    "INVOICE_DATE",
    "INVOICE_DUE_DATE",
    "SELLER_NAME",
    "SELLER_ADDRESS",
    "CLIENT_NAME",
    "CLIENT_ADDRESS",
    "TAX",
    "DISCOUNT",
    "TOTAL",
    "PAYMENT_DUE_DATE",
    "BANK_NAME",
    "ACCOUNT_NUMBER",
    "PAYMENT_METHOD",
]

BASELINE_ENTITIES = [
    "INVOICE_NUMBER",
    "INVOICE_DATE",
    "SELLER_NAME",
    "SELLER_ADDRESS",
    "CLIENT_NAME",
    "CLIENT_ADDRESS",
    "TAX",
    "TOTAL",
]

ENTITY_TYPES = BASELINE_ENTITIES

LABELS = ["O"]

for entity in ENTITY_TYPES:
    LABELS.extend([
        f"B-{entity}",
        f"I-{entity}",
    ])

label2id = {
    label: index
    for index, label in enumerate(LABELS)
}

id2label = {
    index: label
    for label, index in label2id.items()
}

print(LABELS)

['O', 'B-INVOICE_NUMBER', 'I-INVOICE_NUMBER', 'B-INVOICE_DATE', 'I-INVOICE_DATE', 'B-SELLER_NAME', 'I-SELLER_NAME', 'B-SELLER_ADDRESS', 'I-SELLER_ADDRESS', 'B-CLIENT_NAME', 'I-CLIENT_NAME', 'B-CLIENT_ADDRESS', 'I-CLIENT_ADDRESS', 'B-TAX', 'I-TAX', 'B-TOTAL', 'I-TOTAL']


## How it works?

1. Define, what entities I am extracing (let us start with INVOICE_NUMBER only)
2. Find this sequence in the initial OCR text (find spans function does that)
3. Get the coordinates of such substring
4. Tokenize _this particular substring_ (separately from the whole string)
5. Label this substring with BIO (how? what does it mean)
6. Named entitites (What is this?)
7. Structureed invoice JSON - you mean, generation of structured json?

## Full pipeline

### Invoice model

In [12]:
@dataclass(frozen=True, slots=True)
class CandidateSpan:
    start: int
    end: int

    def overlaps(
        self,
        other: "CandidateSpan",
    ) -> bool:
        return (
            self.start < other.end
            and self.end > other.start
        )

In [13]:
def normalize_with_mapping(text):
    normalized = []
    original_positions = []

    previous_was_space = True

    for index, character in enumerate(str(text)):
        if character.isalnum():
            normalized.append(character.lower())
            original_positions.append(index)
            previous_was_space = False

        elif not previous_was_space:
            normalized.append(" ")
            original_positions.append(index)
            previous_was_space = True

    if normalized and normalized[-1] == " ":
        normalized.pop()
        original_positions.pop()

    return "".join(normalized), original_positions

In [14]:
class Field(str):
    @classmethod
    def __get_pydantic_core_schema__(
        cls,
        source_type,
        handler,
    ):
        return (
            core_schema
            .no_info_after_validator_function(
                cls,
                core_schema.str_schema(),
            )
        )

    def normalized(self) -> str:
        value = unicodedata.normalize(
            "NFKC",
            self,
        )

        return re.sub(
            r"\s+",
            " ",
            value,
        ).strip()
    
    def alignment_parts(
        self,
    ) -> tuple["Field", ...]:
        return (self,)

    def find_matches(
        self,
        text: str,
    ) -> list[CandidateSpan]:
        normalized_text, positions = (
            normalize_with_mapping(text)
        )
        normalized_value, _ = (
            normalize_with_mapping(self)
        )

        if not normalized_value:
            return []

        matches = []
        search_start = 0

        while True:
            normalized_start = normalized_text.find(
                normalized_value,
                search_start,
            )

            if normalized_start == -1:
                break

            normalized_end = (
                normalized_start
                + len(normalized_value)
            )

            matches.append(
                CandidateSpan(
                    start=positions[normalized_start],
                    end=(
                        positions[normalized_end - 1]
                        + 1
                    ),
                )
            )

            search_start = normalized_start + 1

        return matches


class MoneyField(Field):
    def digits(self) -> str:
        return "".join(
            re.findall(r"\d", self)
        )

    def find_matches(
        self,
        text: str,
    ) -> list[CandidateSpan]:
        digits = self.digits()

        if not digits:
            return []

        pattern = (
            r"(?<!\d)"
            + r"[\s,.'’]*".join(
                re.escape(digit)
                for digit in digits
            )
            + r"(?!\d)"
        )

        return [
            CandidateSpan(
                start=match.start(),
                end=match.end(),
            )
            for match in re.finditer(
                pattern,
                text,
            )
        ]

class IdentifierField(Field):
    def compact(self) -> str:
        return re.sub(
            r"[^A-Za-z0-9]",
            "",
            self,
        ).upper()

    def find_matches(
        self,
        text: str,
    ) -> list[CandidateSpan]:
        characters = re.findall(
            r"[A-Za-z0-9]",
            self,
        )

        if not characters:
            return []

        pattern = (
            r"(?<![A-Za-z0-9])"
            + r"[^A-Za-z0-9]*".join(
                re.escape(character)
                for character in characters
            )
            + r"(?![A-Za-z0-9])"
        )

        return [
            CandidateSpan(
                start=match.start(),
                end=match.end(),
            )
            for match in re.finditer(
                pattern,
                text,
                flags=re.IGNORECASE,
            )
        ]

class AddressField(Field):
    def alignment_parts(self) -> tuple["AddressField", ...]:
        return tuple(
            type(self)(line.strip())
            for line in self.splitlines()
            if line.strip()
        )

In [15]:
class EntityType(StrEnum):
    INVOICE_NUMBER = "INVOICE_NUMBER"
    INVOICE_DATE = "INVOICE_DATE"
    INVOICE_DUE_DATE = "INVOICE_DUE_DATE"

    SELLER_NAME = "SELLER_NAME"
    SELLER_ADDRESS = "SELLER_ADDRESS"

    CLIENT_NAME = "CLIENT_NAME"
    CLIENT_ADDRESS = "CLIENT_ADDRESS"

    TAX = "TAX"
    TOTAL = "TOTAL"
    DISCOUNT = "DISCOUNT"

    PAYMENT_DUE_DATE = "PAYMENT_DUE_DATE"
    ACCOUNT_NUMBER = "ACCOUNT_NUMBER"
    BANK_NAME = "BANK_NAME"
    PAYMENT_METHOD = "PAYMENT_METHOD"
    


class InvoiceData(BaseModel):
    invoice_number: IdentifierField | None = None
    invoice_date: Field | None = None
    invoice_due_date: Field | None = None
    seller_name: Field | None = None
    seller_address: AddressField | None = None
    client_name: Field | None = None
    client_address: AddressField | None = None


class SubtotalData(BaseModel):
    tax: MoneyField | None = None
    total: MoneyField | None = None
    discount: MoneyField | None = None


class PaymentInstructions(BaseModel):
    due_date: Field | None = None
    account_number: IdentifierField | None = None
    bank_name: Field | None = None
    payment_method: Field | None = None


class InvoiceTarget(BaseModel):
    invoice: InvoiceData
    subtotal: SubtotalData
    payment_instructions: PaymentInstructions
    items: list[dict]

    def extract_fields(self) -> list[tuple[EntityType, Field]]:
        fields = [
            ( EntityType.INVOICE_NUMBER, self.invoice.invoice_number ),
            ( EntityType.INVOICE_DATE, self.invoice.invoice_date ),
            ( EntityType.INVOICE_DUE_DATE, self.invoice.invoice_due_date ),
            ( EntityType.SELLER_NAME, self.invoice.seller_name ),
            ( EntityType.SELLER_ADDRESS, self.invoice.seller_address ),
            ( EntityType.CLIENT_NAME, self.invoice.client_name ),
            ( EntityType.CLIENT_ADDRESS, self.invoice.client_address ),
            ( EntityType.TAX, self.subtotal.tax ),
            ( EntityType.DISCOUNT, self.subtotal.discount ),
            ( EntityType.TOTAL, self.subtotal.total ),
            ( EntityType.PAYMENT_DUE_DATE, self.payment_instructions.due_date ),
            ( EntityType.BANK_NAME, self.payment_instructions.bank_name ),
            ( EntityType.ACCOUNT_NUMBER, self.payment_instructions.account_number ),
            ( EntityType.PAYMENT_METHOD, self.payment_instructions.payment_method )
        ]
        return [
            (entity, value)
            for entity, value in fields
            if value is not None
        ]

@dataclass(frozen=True, slots=True)
class EntitySpan:
    """
    Represents entity span for each particular entity
    """

    start: int
    end: int
    entity: str
    value: str

    def __post_init__(self):
        if self.start < 0:
            raise ValueError(
                "Span start cannot be negative"
            )

        if self.end <= self.start:
            raise ValueError(
                f"Invalid span: [{self.start}, {self.end})"
            )

        if not self.entity:
            raise ValueError(
                "Entity type cannot be empty"
            )

    @property
    def length(self) -> int:
        return self.end - self.start

    def overlaps(
        self,
        other: "EntitySpan",
    ) -> bool:
        return (
            self.start < other.end
            and self.end > other.start
        )

    def overlaps_range(
        self,
        start: int,
        end: int,
    ) -> bool:
        return (
            self.start < end
            and self.end > start
        )

### Candidates selector

In [16]:
@dataclass
class CandidatesSelector:
    def _select_unique_candidate(
        self,
        text: str,
        candidates: list[CandidateSpan],
    ) -> CandidateSpan | None:
        if len(candidates) != 1:
            return None

        return candidates[0]

    def _select_final_total_candidate(
        self,
        text: str,
        candidates: list[CandidateSpan],
    ) -> CandidateSpan | None:
        if not candidates:
            return None

        anchors = list(
            re.finditer(
                r"\btotal\b",
                text,
                flags=re.IGNORECASE,
            )
        )

        if anchors:
            final_anchor = anchors[-1].start()

            candidates_after_total = [
                candidate
                for candidate in candidates
                if candidate.start > final_anchor
            ]

            if candidates_after_total:
                return candidates_after_total[-1]

        return candidates[-1]

    def select(
        self,
        text: str,
        entity_type: EntityType,
        candidates: list[CandidateSpan]
    ) -> CandidateSpan | None:
        if entity_type in {EntityType.TAX, EntityType.TOTAL}:
            return self._select_final_total_candidate(
                text=text,
                candidates=candidates
            )
        return self._select_unique_candidate(
            text=text,
            candidates=candidates
        )

#### Test

In [17]:
selector = CandidatesSelector()

##### 1. A regular entity with one candidate

In [18]:
text = "Invoice: 12345"

candidates = [
    CandidateSpan(
        start=9,
        end=14,
    ),
]

selected = selector.select(
    text=text,
    entity_type=(
        EntityType.INVOICE_NUMBER
    ),
    candidates=candidates,
)

assert selected == CandidateSpan(
    start=9,
    end=14,
)

assert text[
    selected.start:selected.end
] == "12345"

print(
    "Unique candidate:",
    selected,
    text[selected.start:selected.end],
)

Unique candidate: CandidateSpan(start=9, end=14) 12345


##### 2. A regular entity with multiple candidates.

The selector cannot make a safe choice.

In [19]:
text = "Ref: 123 Ref: 123"

candidates = [
    CandidateSpan(
        start=5,
        end=8,
    ),
    CandidateSpan(
        start=14,
        end=17,
    ),
]

selected = selector.select(
    text=text,
    entity_type=(
        EntityType.INVOICE_NUMBER
    ),
    candidates=candidates,
)

assert selected is None

print(
    "Ambiguous regular entity:",
    selected,
)

Ambiguous regular entity: None


##### 3. Repeated TOTAL value

The selector chooses the candidate after "Total".

In [20]:
text = "110.00 Total 110.00"

candidates = [
    CandidateSpan(
        start=0,
        end=6,
    ),
    CandidateSpan(
        start=13,
        end=19,
    ),
]

selected = selector.select(
    text=text,
    entity_type=EntityType.TOTAL,
    candidates=candidates,
)

assert selected == CandidateSpan(
    start=13,
    end=19,
)

assert text[
    selected.start:selected.end
] == "110.00"

print(
    "Selected total:",
    selected,
    text[selected.start:selected.end],
)


Selected total: CandidateSpan(start=13, end=19) 110.00


##### 4  No candidates.

In [21]:
selected = selector.select(
    text="Invoice without a total",
    entity_type=EntityType.TOTAL,
    candidates=[],
)

assert selected is None

print(
    "No candidates:",
    selected,
)


print(
    "\nAll CandidatesSelector tests passed."
)

No candidates: None

All CandidatesSelector tests passed.


### Document aligner

In [22]:
@dataclass(frozen=True, slots=True)
class FinancialSummarySpans:
    net: CandidateSpan
    tax: CandidateSpan
    total: CandidateSpan

    def get(
        self,
        entity: EntityType,
    ) -> CandidateSpan | None:
        match entity:
            case EntityType.TAX:
                return self.tax

            case EntityType.TOTAL:
                return self.total

            case _:
                return None


class FinancialSummaryAligner:
    MONEY_PATTERN = re.compile(
        r"""
        (?<![\dA-Za-z])
        (
            (?:
                \d{1,3}
                (?:[ \u00A0.'’]\d{3})+
                |
                \d+
            )
            [,.]
            \d{2}
        )
        (?!\d)
        """,
        flags=re.VERBOSE,
    )

    TOTAL_PATTERN = re.compile(
        r"\btotal\b",
        flags=re.IGNORECASE,
    )

    def __init__(
        self,
        lookahead: int = 200,
    ):
        self.lookahead = lookahead

    def find(
        self,
        text: str,
    ) -> FinancialSummarySpans | None:
        total_anchors = list(
            self.TOTAL_PATTERN.finditer(text)
        )

        # Start from the final occurrence because "total" can
        # appear in product descriptions or other sections.
        for anchor in reversed(total_anchors):
            region_end = min(
                len(text),
                anchor.end() + self.lookahead,
            )

            amounts = list(
                self.MONEY_PATTERN.finditer(
                    text,
                    anchor.end(),
                    region_end,
                )
            )

            if len(amounts) < 3:
                continue

            net_match = amounts[0]
            tax_match = amounts[1]
            total_match = amounts[2]

            return FinancialSummarySpans(
                net=CandidateSpan(
                    start=net_match.start(),
                    end=net_match.end(),
                ),
                tax=CandidateSpan(
                    start=tax_match.start(),
                    end=tax_match.end(),
                ),
                total=CandidateSpan(
                    start=total_match.start(),
                    end=total_match.end(),
                ),
            )

        return None

@dataclass
class DocumentAligner:
    financial_summary_aligner: FinancialSummaryAligner
    selector: CandidatesSelector

    def find_spans(self, text: str, target: InvoiceTarget) -> list[EntitySpan]:
        
        financial_summary = (
            self.financial_summary_aligner.find(text)
        )
        spans: list[EntitySpan] = []
        occupied: list[CandidateSpan] = []

        for entity, value in target.extract_fields():

            for value_part in value.alignment_parts():
                financial_candidate = (
                    financial_summary.get(entity)
                    if financial_summary is not None
                    else None
                )
                if financial_candidate is not None:
                    # For TAX and TOTAL, the position in the final
                    # summary row is more reliable than ordinary
                    # text matching.
                    candidates = [financial_candidate]
                else:
                    candidates = value_part.find_matches(text)

                available_candidates = [
                    candidate
                    for candidate in candidates
                    if not any(
                        candidate.overlaps(existing)
                        for existing in occupied
                    )
                ]

                if not available_candidates:
                    continue

                if financial_candidate is not None:
                    # The financial aligner already selected the
                    # semantic position.
                    selected = available_candidates[0]
                else:
                    selected = self.selector.select(
                        text=text,
                        entity_type=entity,
                        candidates=available_candidates
                    )

                if selected is None:
                    continue

                spans.append(
                    EntitySpan(
                        start=selected.start,
                        end=selected.end,
                        entity=entity,
                        # Preserve the OCR-observed text rather than
                        # copying the expected JSON value.
                        value=text[
                            selected.start:selected.end
                        ],
                    )
                )

                occupied.append(selected)

        return sorted(
            spans,
            key=lambda span: span.start,
        )

#### Test

In [23]:
row = df.iloc[2]
text = row.ocred_text

In [24]:
document_aligner = DocumentAligner(
    financial_summary_aligner=(
        FinancialSummaryAligner()
    ),
    selector=CandidatesSelector(),
)

In [25]:
target = InvoiceTarget(
    invoice=InvoiceData(
        invoice_number="40108666",
        invoice_date="02/07/2020",
        seller_name=(
            "Bailey, Murray and Lewis"
        ),
        seller_address=(
            "33771 Powell Pike Suite 054\n"
            "East Lisastad, RI 21938"
        ),
        client_name="Hester Inc",
        client_address=(
            "4000 Tonya Rapid Suite 963\n"
            "South Marissaport, MN 39537"
        ),
    ),
    subtotal=SubtotalData(
        tax="45.29",
        total="498.21",
    ),
    payment_instructions=(
        PaymentInstructions(
            account_number=(
                "GBBOMTNI52501633185191"
            ),
        )
    ),
    items=[],
)

In [26]:
spans = document_aligner.find_spans(
    text=text,
    target=target,
)

##### 1. Complete document alignment

In [27]:
expected = [
    (
        EntityType.INVOICE_NUMBER,
        "40108666",
    ),
    (
        EntityType.INVOICE_DATE,
        "02/07/2020",
    ),
    (
        EntityType.SELLER_NAME,
        "Bailey, Murray and Lewis",
    ),
    (
        EntityType.CLIENT_NAME,
        "Hester Inc",
    ),
    (
        EntityType.SELLER_ADDRESS,
        "33771 Powell Pike Suite 054",
    ),
    (
        EntityType.CLIENT_ADDRESS,
        "4000 Tonya Rapid Suite 963",
    ),
    (
        EntityType.SELLER_ADDRESS,
        "East Lisastad, RI 21938",
    ),
    (
        EntityType.CLIENT_ADDRESS,
        "South Marissaport, MN 39537",
    ),
    (
        EntityType.ACCOUNT_NUMBER,
        "GBBOMTNI52501633185191",
    ),
    (
        EntityType.TAX,
        "45,29",
    ),
    (
        EntityType.TOTAL,
        "498,21",
    ),
]


actual = [
    (
        span.entity,
        span.value,
    )
    for span in spans
]


assert actual == expected, (
    "DocumentAligner returned unexpected spans.\n\n"
    f"Expected:\n{expected}\n\n"
    f"Actual:\n{actual}"
)

print(
    "Test 1 passed: all expected entities "
    "were aligned correctly."
)

Test 1 passed: all expected entities were aligned correctly.


##### Test 2: coordinates point to the correct OCR text

In [28]:
for span in spans:
    value_from_text = text[
        span.start:span.end
    ]

    assert value_from_text == span.value, (
        f"Invalid coordinates for "
        f"{span.entity}.\n"
        f"Span value: {span.value!r}\n"
        f"Text value: {value_from_text!r}\n"
        f"Coordinates: "
        f"[{span.start}, {span.end})"
    )

print(
    "Test 2 passed: all coordinates "
    "point to the correct OCR substrings."
)

Test 2 passed: all coordinates point to the correct OCR substrings.


##### Test 3: spans are sorted

In [29]:
starts = [
    span.start
    for span in spans
]

assert starts == sorted(starts), (
    "DocumentAligner did not return "
    "spans in document order."
)

print(
    "Test 3 passed: spans are sorted "
    "in document order."
)

Test 3 passed: spans are sorted in document order.


##### Test 4: spans do not overlap

In [30]:
for previous, current in zip(
    spans,
    spans[1:],
):
    assert not previous.overlaps(
        current
    ), (
        "Found overlapping spans:\n"
        f"{previous}\n"
        f"{current}"
    )

print(
    "Test 4 passed: spans do not overlap."
)

Test 4 passed: spans do not overlap.


##### Test 5: TAX and TOTAL come from the final row

The same values occur before and after the final `Total` label. The expected spans should be after that label.

In [31]:
final_total_anchor = text.rfind(
    "Total"
)

tax_span = next(
    span
    for span in spans
    if span.entity == EntityType.TAX
)

total_span = next(
    span
    for span in spans
    if span.entity == EntityType.TOTAL
)


assert tax_span.start > final_total_anchor
assert total_span.start > final_total_anchor

assert tax_span.value == "45,29"
assert total_span.value == "498,21"

print(
    "Test 5 passed: TAX and TOTAL were "
    "selected from the final Total row."
)

Test 5 passed: TAX and TOTAL were selected from the final Total row.


##### Test 6: account number is extracted correctly

In [32]:
account_spans = [
    span
    for span in spans
    if (
        span.entity
        == EntityType.ACCOUNT_NUMBER
    )
]

assert len(account_spans) == 1

account_span = account_spans[0]

assert account_span.value == (
    "GBBOMTNI52501633185191"
)

assert text[
    account_span.start:
    account_span.end
] == "GBBOMTNI52501633185191"

print(
    "Test 6 passed: account number "
    "was aligned correctly."
)

Test 6 passed: account number was aligned correctly.


##### Display results

In [33]:
spans_df = pd.DataFrame([
    {
        "entity": span.entity,
        "start": span.start,
        "end": span.end,
        "value": span.value,
        "context": text[
            max(0, span.start - 20):
            min(len(text), span.end + 20)
        ],
    }
    for span in spans
])
spans_df

,entity,start,end,value,context
0,INVOICE_NUMBER,12,20,40108666,Invoice no: 40108666 Date of issue: 02/0
1,INVOICE_DATE,36,46,02/07/2020,8666 Date of issue: 02/07/2020 Seller: Client:...
2,SELLER_NAME,63,87,"Bailey, Murray and Lewis","020 Seller: Client: Bailey, Murray and Lewis H..."
3,CLIENT_NAME,88,98,Hester Inc,"y, Murray and Lewis Hester Inc 33771 Powell Pi..."
4,SELLER_ADDRESS,99,126,33771 Powell Pike Suite 054,nd Lewis Hester Inc 33771 Powell Pike Suite 05...
5,CLIENT_ADDRESS,127,153,4000 Tonya Rapid Suite 963,well Pike Suite 054 4000 Tonya Rapid Suite 963...
6,SELLER_ADDRESS,154,177,"East Lisastad, RI 21938","nya Rapid Suite 963 East Lisastad, RI 21938 So..."
7,CLIENT_ADDRESS,178,205,"South Marissaport, MN 39537","Lisastad, RI 21938 South Marissaport, MN 3953..."
8,ACCOUNT_NUMBER,252,274,GBBOMTNI52501633185191,: 998-98-6887 IBAN: GBBOMTNI52501633185191 ITE...
9,TAX,737,742,"45,29","21 Total $ 452,92 $ 45,29 $ 498,21"


### Reporting

In [34]:
from collections.abc import Iterable
from collections.abc import Hashable

def group_spans_by_entity(
    spans: Iterable[EntitySpan],
) -> dict[
    EntityType,
    tuple[EntitySpan, ...],
]:
    grouped: dict[
        EntityType,
        list[EntitySpan],
    ] = defaultdict(list)

    for span in spans:
        grouped[span.entity].append(span)

    return {
        entity: tuple(
            sorted(
                entity_spans,
                key=lambda span: span.start,
            )
        )
        for entity, entity_spans
        in grouped.items()
    }

def money_digits(value: object) -> str:
    return "".join(
        character
        for character in str(value)
        if character.isdigit()
    )

@dataclass
class DocumentReportRow:
    row_index: Hashable
    file_name: str
    expected_entities: int
    matched_entities: int
    alignment_coverage: float
    missing_entities: tuple[str, ...]

    @classmethod
    def create(
        cls,
        *,
        row_index: Hashable,
        file_name: str,
        expected_fields: tuple[
            tuple[EntityType, Field],
            ...
        ],
        spans: tuple[EntitySpan, ...],
    ) -> "DocumentReportRow":
        expected = {
            entity
            for entity, _ in expected_fields
        }

        matched = {
            span.entity
            for span in spans
        }

        matched_expected = expected & matched

        return cls(
            row_index=row_index,
            file_name=file_name,
            expected_entities=len(expected),
            matched_entities=len(
                matched_expected
            ),
            alignment_coverage=(
                len(matched_expected) / len(expected)
                if expected
                else 1.0
            ),
            missing_entities=tuple(
                entity.value
                for entity in sorted(
                    expected - matched,
                    key=lambda item: item.value,
                )
            ),
        )


@dataclass
class DetailReportRow:
    row_index: Hashable
    file_name: str
    entity: str
    expected: str
    observed: str | None
    aligned: bool
    span_count: int
    parts: int
    matched_parts: int
    fully_raw_matched: bool
    partially_raw_matched: bool
    ambiguous: bool
    candidate_counts: tuple[int, ...]
    selected_spans: tuple[EntitySpan, ...]

    @classmethod
    def create(
        cls,
        *,
        row_index: Hashable,
        file_name: str,
        text: str,
        entity: EntityType,
        value: Field,
        selected_spans: tuple[
            EntitySpan,
            ...
        ],
    ) -> "DetailReportRow":
        value_parts = value.alignment_parts()

        part_candidates = [
            part.find_matches(text)
            for part in value_parts
        ]

        matched_parts = sum(
            bool(candidates)
            for candidates in part_candidates
        )

        candidate_counts = tuple(
            len(candidates)
            for candidates in part_candidates
        )

        return cls(
            row_index=row_index,
            file_name=file_name,
            entity=entity.value,
            expected=str(value),
            observed=(
                "\n".join(
                    span.value
                    for span in selected_spans
                )
                if selected_spans
                else None
            ),
            aligned=bool(selected_spans),
            span_count=len(selected_spans),
            parts=len(value_parts),
            matched_parts=matched_parts,
            fully_raw_matched=(
                matched_parts == len(value_parts)
            ),
            partially_raw_matched=(
                0 < matched_parts < len(value_parts)
            ),
            ambiguous=any(
                count > 1
                for count in candidate_counts
            ),
            candidate_counts=candidate_counts,
            selected_spans=selected_spans,
        )

@dataclass
class EntityReport:
    df: pd.DataFrame

    def build(self) -> pd.DataFrame:
        if self.df.empty:
            return pd.DataFrame()

        result = (
            self.df
            .groupby("entity", as_index=False)
            .agg(
                expected_documents=(
                    "entity",
                    "size",
                ),
                aligned_documents=(
                    "aligned",
                    "sum",
                ),
                fully_raw_matched=(
                    "fully_raw_matched",
                    "sum",
                ),
                partially_raw_matched=(
                    "partially_raw_matched",
                    "sum",
                ),
                ambiguous_documents=(
                    "ambiguous",
                    "sum",
                ),
            )
        )

        result["alignment_coverage"] = (
            result["aligned_documents"]
            / result["expected_documents"]
        )

        result["raw_match_rate"] = (
            result["fully_raw_matched"]
            / result["expected_documents"]
        )

        result["partial_rate"] = (
            result["partially_raw_matched"]
            / result["expected_documents"]
        )

        result["ambiguous_rate"] = (
            result["ambiguous_documents"]
            / result["expected_documents"]
        )

        return result.sort_values(
            "alignment_coverage"
        )

@dataclass
class FinancialReport:
    df: pd.DataFrame

    def build(self) -> pd.DataFrame:
        if self.df.empty:
            return pd.DataFrame()

        financial_details = self.df[
            self.df["entity"].isin(
                [
                    EntityType.TAX.value,
                    EntityType.TOTAL.value,
                ]
            )
        ].copy()

        if financial_details.empty:
            return pd.DataFrame()

        financial_details["value_matches_json"] = (
            financial_details.apply(
                lambda row: (
                    row["observed"] is not None
                    and money_digits(
                        row["observed"]
                    )
                    == money_digits(
                        row["expected"]
                    )
                ),
                axis=1,
            )
        )

        result = (
            financial_details
            .groupby("entity", as_index=False)
            .agg(
                documents=("entity", "size"),
                spans_found=("aligned", "sum"),
                values_matching_json=(
                    "value_matches_json",
                    "sum",
                ),
            )
        )

        result["span_coverage"] = (
            result["spans_found"]
            / result["documents"]
        )

        result["ocr_value_accuracy"] = (
            result["values_matching_json"]
            / result["documents"]
        )

        return result

In [35]:
from pandas.core.frame import DataFrame

@dataclass(slots=True)
class AlignmentReport:
    documents: pd.DataFrame
    details: pd.DataFrame
    entities: pd.DataFrame
    financial: pd.DataFrame

    @classmethod
    def build(
        cls,
        frame: pd.DataFrame,
        aligner: DocumentAligner,
    ) -> "AlignmentReport":
        document_rows = []
        detail_rows = []

        for row_index, row in frame.iterrows():
            text = str(row["ocred_text"])
            target = cls._get_target(row)
            file_name = row.get(
                "file_name",
                str(row_index),
            )

            expected_fields = tuple(
                target.extract_fields()
            )

            spans = tuple(
                aligner.find_spans(
                    text=text,
                    target=target,
                )
            )

            spans_by_entity = (
                group_spans_by_entity(spans)
            )

            document_rows.append(
                DocumentReportRow.create(
                    row_index=row_index,
                    file_name=file_name,
                    expected_fields=expected_fields,
                    spans=spans,
                )
            )

            detail_rows.extend(
                DetailReportRow.create(
                    row_index=row_index,
                    file_name=file_name,
                    text=text,
                    entity=entity,
                    value=value,
                    selected_spans=(
                        spans_by_entity.get(entity, ())
                    ),
                )
                for entity, value in expected_fields
            )

        documents = pd.DataFrame(
            document_rows
        )

        details = pd.DataFrame(
            detail_rows
        )
    
        entities: DataFrame = EntityReport(df=details).build()

        financial = FinancialReport(df=details).build()

        return cls(
            documents=documents,
            details=details,
            entities=entities,
            financial=financial,
        )

    @staticmethod
    def _get_target(
        row: pd.Series,
    ) -> InvoiceTarget:
        target = row.get("target")

        if isinstance(target, InvoiceTarget):
            return target

        if isinstance(target, dict):
            return InvoiceTarget.model_validate(
                target
            )

        json_data = row["json_data"]

        if isinstance(json_data, str):
            return (
                InvoiceTarget
                .model_validate_json(json_data)
            )

        return InvoiceTarget.model_validate(
            json_data
        )

    def missing_counts(self) -> pd.DataFrame:
        missing = (
            self.details.loc[
                ~self.details["aligned"],
                "entity",
            ]
            .value_counts()
            .rename("missing_count")
        )

        return missing.to_frame()

    def low_coverage(
        self,
        limit: int = 10,
    ) -> pd.DataFrame:
        return self.documents.nsmallest(
            limit,
            "alignment_coverage",
        )

#### Test

In [36]:
row = df.iloc[7]
text = row['ocred_text']
text

'Invoice no: 20012449 Date of issue: 04/22/2013 Seller: Client: Jackson Ltd Combs Inc 58538 Brandon Union Apt. 536 433 Ferguson Land Apt: 377 Taylorshire, PA 90588 Kirkstad, MN 16124 Tax Id: 989-83-1608 Tax Id: 940-79-9180 IBAN: GBIZKVLY05129120946380 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth Press Wine 20L Fruit Cider 1,00 each 299,00 299,00 10% 328,90 Apple Crusher Juice Grape Stainless Maker Grapes New 2 MARIJUANA LEAF WINE STOPPER 4,00 each 7,20 28,80 10% 31,68 SUMMARY VAT [%] Net worth VAT Gross worth 10% 327,80 32,78 360,58 Total $ 327,80 $ 32,78 $ 360,58'

In [37]:
target = InvoiceTarget(
    invoice=InvoiceData(
        invoice_number="20012449",
        invoice_date="04/22/2013",
        seller_name="Jackson Ltd",
        seller_address=(
            "58538 Brandon Union Apt. 536\n"
            "Taylorshire, PA 90588"
        ),
        client_name="Combs Inc",
        client_address=(
            "433 Ferguson Land Apt. 377\n"
            "Kirkstad, MN 16124"
        ),
    ),
    subtotal=SubtotalData(
        tax="32.78",
        total="360.58",
    ),
    payment_instructions=(
        PaymentInstructions(
            account_number=(
                "GBIZKVLY05129120946380"
            ),
        )
    ),
    items=[],
)


test_frame = pd.DataFrame([
    {
        "file_name": "test-invoice.jpg",
        "ocred_text": text,
        "target": target,
    }
])

In [38]:
document_aligner = DocumentAligner(
    financial_summary_aligner=(
        FinancialSummaryAligner()
    ),
    selector=CandidatesSelector(),
)


report = AlignmentReport.build(
    frame=test_frame,
    aligner=document_aligner,
)

##### Test 1: report tables were created

This target contains nine entity types:

1. Invoice number
2. Invoice date
3. Seller name
4. Seller address
5. Client name
6. Client address
7. Tax
8. Total
9. Account number

In [39]:
assert len(report.documents) == 1

assert len(report.details) == 9, (
    "Expected one detail row for each "
    "of the nine target fields"
)

assert not report.entities.empty

assert not report.financial.empty

print(
    "Test 1 passed: all report tables "
    "were created."
)

Test 1 passed: all report tables were created.


##### Test 2: document-level report is correct

In [40]:
document_row = report.documents.iloc[0]


assert document_row["file_name"] == (
    "test-invoice.jpg"
)

assert document_row[
    "expected_entities"
] == 9

assert document_row[
    "matched_entities"
] == 9

assert document_row[
    "alignment_coverage"
] == 1.0

assert document_row[
    "missing_entities"
] == ()


print(
    "Test 2 passed: document alignment "
    "coverage is 100%."
)

Test 2 passed: document alignment coverage is 100%.


##### Test 3: invoice-number details are correct

In [41]:
invoice_number_row = (
    report.details.loc[
        report.details["entity"]
        == EntityType.INVOICE_NUMBER.value
    ]
    .iloc[0]
)


assert invoice_number_row[
    "expected"
] == "20012449"

assert invoice_number_row[
    "observed"
] == "20012449"

assert bool(
    invoice_number_row["aligned"]
)

assert invoice_number_row[
    "span_count"
] == 1

assert invoice_number_row[
    "parts"
] == 1

assert invoice_number_row[
    "matched_parts"
] == 1

assert bool(
    invoice_number_row[
        "fully_raw_matched"
    ]
)

assert not bool(
    invoice_number_row["ambiguous"]
)


print(
    "Test 3 passed: invoice-number "
    "report details are correct."
)

Test 3 passed: invoice-number report details are correct.


##### Test 4: multiline addresses are reported correctly

In [42]:
seller_address_row = (
    report.details.loc[
        report.details["entity"]
        == EntityType.SELLER_ADDRESS.value
    ]
    .iloc[0]
)


assert seller_address_row[
    "expected"
] == (
    "58538 Brandon Union Apt. 536\n"
    "Taylorshire, PA 90588"
)

assert seller_address_row[
    "observed"
] == (
    "58538 Brandon Union Apt. 536\n"
    "Taylorshire, PA 90588"
)

assert seller_address_row[
    "span_count"
] == 2

assert seller_address_row[
    "parts"
] == 2

assert seller_address_row[
    "matched_parts"
] == 2

assert seller_address_row[
    "candidate_counts"
] == (1, 1)

assert bool(
    seller_address_row["aligned"]
)

assert bool(
    seller_address_row[
        "fully_raw_matched"
    ]
)


print(
    "Test 4 passed: seller-address lines "
    "were combined correctly."
)

Test 4 passed: seller-address lines were combined correctly.


The client address deliberately has an OCR punctuation difference:

```
Expected: Apt. 377
Observed: Apt: 377
```

In [43]:
client_address_row = (
    report.details.loc[
        report.details["entity"]
        == EntityType.CLIENT_ADDRESS.value
    ]
    .iloc[0]
)


assert client_address_row[
    "expected"
] == (
    "433 Ferguson Land Apt. 377\n"
    "Kirkstad, MN 16124"
)

assert client_address_row[
    "observed"
] == (
    "433 Ferguson Land Apt: 377\n"
    "Kirkstad, MN 16124"
)

assert client_address_row[
    "span_count"
] == 2

assert client_address_row[
    "parts"
] == 2

assert client_address_row[
    "matched_parts"
] == 2

assert bool(
    client_address_row["aligned"]
)

print(
    "Test 4b passed: normalized address "
    "matching preserved the OCR punctuation."
)

Test 4b passed: normalized address matching preserved the OCR punctuation.


##### Test 5: account-number report is correct

In [44]:
account_row = (
    report.details.loc[
        report.details["entity"]
        == EntityType.ACCOUNT_NUMBER.value
    ]
    .iloc[0]
)


assert account_row[
    "expected"
] == "GBIZKVLY05129120946380"

assert account_row[
    "observed"
] == "GBIZKVLY05129120946380"

assert bool(
    account_row["aligned"]
)

assert account_row[
    "span_count"
] == 1

assert account_row[
    "candidate_counts"
] == (1,)

assert not bool(
    account_row["ambiguous"]
)


print(
    "Test 5 passed: account-number "
    "report details are correct."
)

Test 5 passed: account-number report details are correct.


##### Test 6: financial report is correct

In [45]:
financial = (
    report.financial
    .set_index("entity")
)


assert set(financial.index) == {
    EntityType.TAX.value,
    EntityType.TOTAL.value,
}


for entity in [
    EntityType.TAX.value,
    EntityType.TOTAL.value,
]:
    row = financial.loc[entity]

    assert row["documents"] == 1
    assert row["spans_found"] == 1

    assert row[
        "values_matching_json"
    ] == 1

    assert row[
        "span_coverage"
    ] == 1.0

    assert row[
        "ocr_value_accuracy"
    ] == 1.0


print(
    "Test 6 passed: TAX and TOTAL "
    "financial metrics are correct."
)

Test 6 passed: TAX and TOTAL financial metrics are correct.


##### Test 7: financial details use OCR values

In [46]:
tax_row = (
    report.details.loc[
        report.details["entity"]
        == EntityType.TAX.value
    ]
    .iloc[0]
)

total_row = (
    report.details.loc[
        report.details["entity"]
        == EntityType.TOTAL.value
    ]
    .iloc[0]
)


assert tax_row["expected"] == "32.78"
assert tax_row["observed"] == "32,78"

assert total_row["expected"] == "360.58"
assert total_row["observed"] == "360,58"


# Both values occur twice:
# once in SUMMARY and once after Total.
assert tax_row[
    "candidate_counts"
] == (2,)

assert total_row[
    "candidate_counts"
] == (2,)

assert bool(
    tax_row["ambiguous"]
)

assert bool(
    total_row["ambiguous"]
)


print(
    "Test 7 passed: financial details "
    "preserve OCR values and report "
    "raw matching ambiguity."
)

Test 7 passed: financial details preserve OCR values and report raw matching ambiguity.


##### Test 8: entity-level report is correct

In [47]:
assert len(report.entities) == 9

assert (
    report.entities[
        "expected_documents"
    ] == 1
).all()

assert (
    report.entities[
        "aligned_documents"
    ] == 1
).all()

assert (
    report.entities[
        "alignment_coverage"
    ] == 1.0
).all()


print(
    "Test 8 passed: all entity types "
    "have 100% alignment coverage."
)


Test 8 passed: all entity types have 100% alignment coverage.


##### Test 9: missing and low-coverage methods

In [48]:
missing = report.missing_counts()

assert missing.empty


low_coverage = report.low_coverage(
    limit=10
)

assert len(low_coverage) == 1

assert low_coverage.iloc[0][
    "alignment_coverage"
] == 1.0


print(
    "Test 9 passed: no missing fields "
    "were reported."
)

Test 9 passed: no missing fields were reported.


##### Display the report

In [49]:
report.documents

,row_index,file_name,expected_entities,matched_entities,alignment_coverage,missing_entities
0,0,test-invoice.jpg,9,9,1.0,()


In [50]:
report.details[
        [
            "entity",
            "expected",
            "observed",
            "aligned",
            "span_count",
            "candidate_counts",
            "ambiguous",
        ]
    ]

,entity,expected,observed,aligned,span_count,candidate_counts,ambiguous
0,INVOICE_NUMBER,20012449,20012449,True,1,"(1,)",False
1,INVOICE_DATE,04/22/2013,04/22/2013,True,1,"(1,)",False
2,SELLER_NAME,Jackson Ltd,Jackson Ltd,True,1,"(1,)",False
3,SELLER_ADDRESS,"58538 Brandon Union Apt. 536\nTaylorshire, PA ...","58538 Brandon Union Apt. 536\nTaylorshire, PA ...",True,2,"(1, 1)",False
4,CLIENT_NAME,Combs Inc,Combs Inc,True,1,"(1,)",False
5,CLIENT_ADDRESS,"433 Ferguson Land Apt. 377\nKirkstad, MN 16124","433 Ferguson Land Apt: 377\nKirkstad, MN 16124",True,2,"(1, 1)",False
6,TAX,32.78,"32,78",True,1,"(2,)",True
7,TOTAL,360.58,"360,58",True,1,"(2,)",True
8,ACCOUNT_NUMBER,GBIZKVLY05129120946380,GBIZKVLY05129120946380,True,1,"(1,)",False


In [51]:
report.entities.style.format({
    "alignment_coverage": "{:.2%}",
    "raw_match_rate": "{:.2%}",
    "partial_rate": "{:.2%}",
    "ambiguous_rate": "{:.2%}",
})

,entity,expected_documents,aligned_documents,fully_raw_matched,partially_raw_matched,ambiguous_documents,alignment_coverage,raw_match_rate,partial_rate,ambiguous_rate
0,ACCOUNT_NUMBER,1,1,1,0,0,100.00%,100.00%,0.00%,0.00%
1,CLIENT_ADDRESS,1,1,1,0,0,100.00%,100.00%,0.00%,0.00%
2,CLIENT_NAME,1,1,1,0,0,100.00%,100.00%,0.00%,0.00%
3,INVOICE_DATE,1,1,1,0,0,100.00%,100.00%,0.00%,0.00%
4,INVOICE_NUMBER,1,1,1,0,0,100.00%,100.00%,0.00%,0.00%
5,SELLER_ADDRESS,1,1,1,0,0,100.00%,100.00%,0.00%,0.00%
6,SELLER_NAME,1,1,1,0,0,100.00%,100.00%,0.00%,0.00%
7,TAX,1,1,1,0,1,100.00%,100.00%,0.00%,100.00%
8,TOTAL,1,1,1,0,1,100.00%,100.00%,0.00%,100.00%


In [52]:
report.financial.style.format({
    "span_coverage": "{:.2%}",
    "ocr_value_accuracy": "{:.2%}",
})

,entity,documents,spans_found,values_matching_json,span_coverage,ocr_value_accuracy
0,TAX,1,1,1,100.00%,100.00%
1,TOTAL,1,1,1,100.00%,100.00%


### Inspection

In [53]:
@dataclass(frozen=True, slots=True)
class AlignmentInspector:
    report: AlignmentReport

    def document(
        self,
        row_index: int,
    ) -> pd.DataFrame:
        return (
            self.report.details.loc[
                self.report.details[
                    "row_index"
                ] == row_index
            ]
            .sort_values("entity")
            .reset_index(drop=True)
        )

    def missing(
        self,
    ) -> pd.DataFrame:
        return (
            self.report.details.loc[
                ~self.report.details["aligned"]
            ]
            .sort_values(
                [
                    "entity",
                    "file_name",
                ]
            )
            .reset_index(drop=True)
        )

    def ambiguous(
        self,
    ) -> pd.DataFrame:
        return (
            self.report.details.loc[
                self.report.details["ambiguous"]
            ]
            .sort_values(
                [
                    "entity",
                    "file_name",
                ]
            )
            .reset_index(drop=True)
        )

    def partial_matches(
        self,
    ) -> pd.DataFrame:
        return (
            self.report.details.loc[
                self.report.details[
                    "partially_raw_matched"
                ]
            ]
            .sort_values(
                [
                    "entity",
                    "file_name",
                ]
            )
            .reset_index(drop=True)
        )

    def worst_documents(
        self,
        limit: int = 10,
    ) -> pd.DataFrame:
        return self.report.low_coverage(
            limit
        )

    def financial_mismatches(
        self,
    ) -> pd.DataFrame:
        financial = self.report.details[
            self.report.details["entity"].isin(
                [
                    EntityType.TAX.value,
                    EntityType.TOTAL.value,
                ]
            )
        ].copy()

        financial["value_matches_json"] = (
            financial.apply(
                lambda row: (
                    row["observed"] is not None
                    and money_digits(
                        row["observed"]
                    )
                    == money_digits(
                        row["expected"]
                    )
                ),
                axis=1,
            )
        )

        return (
            financial.loc[
                ~financial[
                    "value_matches_json"
                ]
            ]
            .reset_index(drop=True)
        )

#### Test

In [54]:
row = df.iloc[11]
text = row['ocred_text']
text

'Invoice no: 35593328 Date of issue: 06/08/2011 Seller: Client: Cruz-Carter Alexander-Arias 11251 Braun Square 63947 Erin Parkway Apt: 073 North Alejandro, UT 73496 Guzmanshire, NY 08857 Tax Id: 938-91-6436 Tax Id: 942-92-5992 IBAN: GB4ZRIFK65000994976015 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth hp all in one desktop computer 4,00 each 205,00 820,00 10% 902,00 touch screen 2 WS Pro Mini Pocket PC Windows 3,00 each 119,00 357,00 10% 392,70 10 Computer Stick Intel Z8350 Quad Core 3 Gaming PC Desktop Computer 1,00 each 999,50 999,50 10% 1 099,45 HDMI RGB Intel i7 RTX 2060, HDD+SSD, Windows 10, Wifi Lenovo C2O-OO..all in one 2,00 each 277,64 555,28 10% 610,81 desktop computer with Wireless Mouse_ SUMMARY VAT [%] Net worth VAT Gross worth 10% 2 731,78 273,18 3 004,96 Total $ 2 731,78 $ 273,18 $ 3 004,96'

In [55]:
target = InvoiceTarget(
    invoice=InvoiceData(
        invoice_number="35593328",
        invoice_date="06/08/2011",
        seller_name="Cruz-Carter",
        seller_address=(
            "11251 Braun Square\n"
            "North Alejandro, UT 73496"
        ),
        client_name="Alexander-Arias",
        client_address=(
            "63947 Erin Parkway Apt. 073\n"
            "Guzmanshire, NY 08857"
        ),
    ),
    subtotal=SubtotalData(
        tax="273.18",
        total="3004.96",
    ),
    payment_instructions=(
        PaymentInstructions(
            account_number=(
                "GB4ZRIFK65000994976015"
            ),
        )
    ),
    items=[],
)


test_frame = pd.DataFrame([
    {
        "file_name": "invoice-35593328.jpg",
        "ocred_text": text,
        "target": target,
    }
])


document_aligner = DocumentAligner(
    financial_summary_aligner=(
        FinancialSummaryAligner()
    ),
    selector=CandidatesSelector(),
)


report = AlignmentReport.build(
    frame=test_frame,
    aligner=document_aligner,
)


inspector = AlignmentInspector(
    report=report
)

##### Test 1: inspect the document

In [56]:
document_details = inspector.document(
    row_index=0
)


expected_entities = {
    EntityType.INVOICE_NUMBER.value,
    EntityType.INVOICE_DATE.value,
    EntityType.SELLER_NAME.value,
    EntityType.SELLER_ADDRESS.value,
    EntityType.CLIENT_NAME.value,
    EntityType.CLIENT_ADDRESS.value,
    EntityType.TAX.value,
    EntityType.TOTAL.value,
    EntityType.ACCOUNT_NUMBER.value,
}


assert len(document_details) == 9

assert set(
    document_details["entity"]
) == expected_entities

assert (
    document_details["aligned"]
).all()

assert (
    document_details["file_name"]
    == "invoice-35593328.jpg"
).all()

assert (
    document_details["row_index"]
    == 0
).all()


print(
    "Test 1 passed: inspector returned "
    "all nine expected entity details."
)

Test 1 passed: inspector returned all nine expected entity details.


##### Test 2: results are sorted by entity

In [57]:
actual_order = (
    document_details[
        "entity"
    ].tolist()
)

expected_order = sorted(
    expected_entities
)


assert actual_order == expected_order, (
    "Document details are not sorted.\n"
    f"Expected: {expected_order}\n"
    f"Actual: {actual_order}"
)


print(
    "Test 2 passed: document details "
    "are sorted by entity."
)

Test 2 passed: document details are sorted by entity.


##### Test 3: invoice information is correct

In [58]:
invoice_number_row = (
    document_details.loc[
        document_details["entity"]
        == EntityType.INVOICE_NUMBER.value
    ]
    .iloc[0]
)

invoice_date_row = (
    document_details.loc[
        document_details["entity"]
        == EntityType.INVOICE_DATE.value
    ]
    .iloc[0]
)


assert invoice_number_row[
    "expected"
] == "35593328"

assert invoice_number_row[
    "observed"
] == "35593328"

assert invoice_date_row[
    "expected"
] == "06/08/2011"

assert invoice_date_row[
    "observed"
] == "06/08/2011"


print(
    "Test 3 passed: invoice number "
    "and date are correct."
)

Test 3 passed: invoice number and date are correct.


##### Test 4: names are correct

In [59]:
seller_name_row = (
    document_details.loc[
        document_details["entity"]
        == EntityType.SELLER_NAME.value
    ]
    .iloc[0]
)

client_name_row = (
    document_details.loc[
        document_details["entity"]
        == EntityType.CLIENT_NAME.value
    ]
    .iloc[0]
)


assert seller_name_row[
    "expected"
] == "Cruz-Carter"

assert seller_name_row[
    "observed"
] == "Cruz-Carter"

assert client_name_row[
    "expected"
] == "Alexander-Arias"

assert client_name_row[
    "observed"
] == "Alexander-Arias"


print(
    "Test 4 passed: seller and client "
    "names are correct."
)

Test 4 passed: seller and client names are correct.


##### Test 5: multiline addresses are correct

In [60]:
seller_address_row = (
    document_details.loc[
        document_details["entity"]
        == EntityType.SELLER_ADDRESS.value
    ]
    .iloc[0]
)


assert seller_address_row[
    "expected"
] == (
    "11251 Braun Square\n"
    "North Alejandro, UT 73496"
)

assert seller_address_row[
    "observed"
] == (
    "11251 Braun Square\n"
    "North Alejandro, UT 73496"
)

assert seller_address_row[
    "parts"
] == 2

assert seller_address_row[
    "matched_parts"
] == 2

assert seller_address_row[
    "span_count"
] == 2

assert seller_address_row[
    "candidate_counts"
] == (1, 1)


client_address_row = (
    document_details.loc[
        document_details["entity"]
        == EntityType.CLIENT_ADDRESS.value
    ]
    .iloc[0]
)


assert client_address_row[
    "expected"
] == (
    "63947 Erin Parkway Apt. 073\n"
    "Guzmanshire, NY 08857"
)

# OCR contains "Apt:" instead of "Apt.".
assert client_address_row[
    "observed"
] == (
    "63947 Erin Parkway Apt: 073\n"
    "Guzmanshire, NY 08857"
)

assert client_address_row[
    "parts"
] == 2

assert client_address_row[
    "matched_parts"
] == 2

assert client_address_row[
    "span_count"
] == 2

assert client_address_row[
    "candidate_counts"
] == (1, 1)

assert bool(
    client_address_row[
        "fully_raw_matched"
    ]
)


print(
    "Test 5 passed: multiline addresses "
    "and OCR punctuation differences "
    "were handled correctly."
)


Test 5 passed: multiline addresses and OCR punctuation differences were handled correctly.


##### Test 6: account number is correct

In [61]:
account_row = (
    document_details.loc[
        document_details["entity"]
        == EntityType.ACCOUNT_NUMBER.value
    ]
    .iloc[0]
)


assert account_row[
    "expected"
] == "GB4ZRIFK65000994976015"

assert account_row[
    "observed"
] == "GB4ZRIFK65000994976015"

assert bool(
    account_row["aligned"]
)

assert account_row[
    "span_count"
] == 1

assert account_row[
    "candidate_counts"
] == (1,)

assert not bool(
    account_row["ambiguous"]
)


print(
    "Test 6 passed: account number "
    "is correct."
)

Test 6 passed: account number is correct.


##### Test 7: missing and partial matches are empty

All expected fields should be fully aligned.

In [62]:
missing = inspector.missing()

partial_matches = (
    inspector.partial_matches()
)


assert missing.empty, (
    "No missing entities were expected, "
    f"but received:\n{missing}"
)

assert partial_matches.empty, (
    "No partial matches were expected, "
    f"but received:\n{partial_matches}"
)


print(
    "Test 7 passed: no missing or "
    "partial matches were reported."
)

Test 7 passed: no missing or partial matches were reported.


##### Test 8: TAX and TOTAL are ambiguous raw matches


In [63]:
ambiguous = inspector.ambiguous()


assert len(ambiguous) == 2

assert set(
    ambiguous["entity"]
) == {
    EntityType.TAX.value,
    EntityType.TOTAL.value,
}

assert (
    ambiguous["ambiguous"]
).all()

assert set(
    ambiguous["candidate_counts"]
) == {
    (2,),
}


tax_row = (
    ambiguous.loc[
        ambiguous["entity"]
        == EntityType.TAX.value
    ]
    .iloc[0]
)

total_row = (
    ambiguous.loc[
        ambiguous["entity"]
        == EntityType.TOTAL.value
    ]
    .iloc[0]
)


assert tax_row[
    "expected"
] == "273.18"

assert tax_row[
    "observed"
] == "273,18"

assert total_row[
    "expected"
] == "3004.96"

assert total_row[
    "observed"
] == "3 004,96"


print(
    "Test 8 passed: TAX and TOTAL "
    "raw-match ambiguities were reported."
)

Test 8 passed: TAX and TOTAL raw-match ambiguities were reported.


##### Test 9: financial values are not mismatches

Although punctuation and thousands separators differ, the values are numerically equivalent.

In [64]:
financial_mismatches = (
    inspector.financial_mismatches()
)


assert financial_mismatches.empty, (
    "Expected no financial mismatches, "
    f"but received:\n"
    f"{financial_mismatches}"
)


print(
    "Test 9 passed: equivalent financial "
    "values were not reported as mismatches."
)

Test 9 passed: equivalent financial values were not reported as mismatches.


##### Test 10: worst document query

There is only one document in this report.

In [65]:
worst_documents = (
    inspector.worst_documents(
        limit=10
    )
)


assert len(worst_documents) == 1

assert worst_documents.iloc[0][
    "file_name"
] == "invoice-35593328.jpg"

assert worst_documents.iloc[0][
    "alignment_coverage"
] == 1.0

assert worst_documents.iloc[0][
    "missing_entities"
] == ()


print(
    "Test 10 passed: worst_documents() "
    "returned the expected document."
)

Test 10 passed: worst_documents() returned the expected document.


##### Test 11: unknown document index

In [66]:
unknown_document = (
    inspector.document(
        row_index=999
    )
)


assert unknown_document.empty


print(
    "Test 11 passed: unknown document "
    "index returned an empty DataFrame."
)

Test 11 passed: unknown document index returned an empty DataFrame.


##### Display results

In [67]:
inspector.document(0)[
    [
        "entity",
        "expected",
        "observed",
        "aligned",
        "span_count",
        "parts",
        "matched_parts",
        "candidate_counts",
        "ambiguous",
    ]
]

,entity,expected,observed,aligned,span_count,parts,matched_parts,candidate_counts,ambiguous
0,ACCOUNT_NUMBER,GB4ZRIFK65000994976015,GB4ZRIFK65000994976015,True,1,1,1,"(1,)",False
1,CLIENT_ADDRESS,"63947 Erin Parkway Apt. 073\nGuzmanshire, NY 0...","63947 Erin Parkway Apt: 073\nGuzmanshire, NY 0...",True,2,2,2,"(1, 1)",False
2,CLIENT_NAME,Alexander-Arias,Alexander-Arias,True,1,1,1,"(1,)",False
3,INVOICE_DATE,06/08/2011,06/08/2011,True,1,1,1,"(1,)",False
4,INVOICE_NUMBER,35593328,35593328,True,1,1,1,"(1,)",False
5,SELLER_ADDRESS,"11251 Braun Square\nNorth Alejandro, UT 73496","11251 Braun Square\nNorth Alejandro, UT 73496",True,2,2,2,"(1, 1)",False
6,SELLER_NAME,Cruz-Carter,Cruz-Carter,True,1,1,1,"(1,)",False
7,TAX,273.18,"273,18",True,1,1,1,"(2,)",True
8,TOTAL,3004.96,"3 004,96",True,1,1,1,"(2,)",True


Ambiguous matches

In [68]:
inspector.ambiguous()[
    [
        "entity",
        "expected",
        "observed",
        "candidate_counts",
    ]
]

,entity,expected,observed,candidate_counts
0,TAX,273.18,"273,18","(2,)"
1,TOTAL,3004.96,"3 004,96","(2,)"


Missing entities

In [69]:
inspector.missing()

,row_index,file_name,entity,expected,observed,aligned,span_count,parts,matched_parts,fully_raw_matched,partially_raw_matched,ambiguous,candidate_counts,selected_spans


Partial matches

In [70]:
inspector.partial_matches()

,row_index,file_name,entity,expected,observed,aligned,span_count,parts,matched_parts,fully_raw_matched,partially_raw_matched,ambiguous,candidate_counts,selected_spans


Financial mismatches

In [71]:
inspector.financial_mismatches()

,row_index,file_name,entity,expected,observed,aligned,span_count,parts,matched_parts,fully_raw_matched,partially_raw_matched,ambiguous,candidate_counts,selected_spans,value_matches_json


### Tokenizer

In [72]:
from collections.abc import Mapping
from dataclasses import dataclass
from typing import TypedDict

import numpy as np
from transformers import PreTrainedTokenizerBase

class TokenizedBatch(TypedDict, total=False):
    input_ids: list[list[int]]
    attention_mask: list[list[int]]
    token_type_ids: list[list[int]]
    labels: list[list[int]]


@dataclass(slots=True)
class TokenLabelBatchEncoder:
    """
    Converts invoice OCR text and its expected JSON into the numerical input
    that BERT needs for token classification training

    OCR text + expected JSON =>
    DocumentAligner finds character positions =>
    Tokenizer splits text into BERT tokens =>
    Each token gets 0, B-ENTITY or I-ENTITY =>
    Text tokens and labels become integer arrays
    """

    # Hugging Face tokenizer
    tokenizer: PreTrainedTokenizerBase
    # Finds where expected invoice values occur in the OCR text.
    aligner: DocumentAligner
    # Converts label names into integers
    label2id: Mapping[str, int]
    # BERT can process only a limited number of tokens at once
    # For standard BERT, this is usually 512 tokens.
    # If an invoice contains more than 512 tokens, 
    # it must be divided into multiple chunks.
    max_length: int = 512
    # Chunks overlap by 64 tokens
    window_stride: int = 64

    def __post_init__(self) -> None:
        if self.max_length <= 0:
            raise ValueError(
                "max_length must be positive"
            )

        if not 0 <= self.window_stride < self.max_length:
            raise ValueError(
                "window_stride must be between 0 "
                "and max_length - 1"
            )

        if "O" not in self.label2id:
            raise ValueError(
                "label2id must contain the O label"
            )

    @staticmethod
    def _get_chunk_range(
        offsets: list[tuple[int, int]],
    ) -> tuple[int, int] | None:
        """
        Receives the character offsets for every token in one chunk.
        An offset tells us where a token came from in the original text.

        Example:
            Text: "Invoice 12345"

            Possible offsets:
            [
                (0, 0),    # [CLS] special token
                (0, 7),    # Invoice
                (8, 11),   # 123
                (11, 13),  # ##45
                (0, 0),    # [SEP] special token
            ]

            (0, 7) means the token uses characters 0 through 6.
        """

        valid_offsets = [
            (start, end)
            for start, end in offsets
            if start != end
        ]

        if not valid_offsets:
            return None

        return (
            valid_offsets[0][0],
            valid_offsets[-1][1],
        )

    @staticmethod
    def _calculate_overlap(
        span: EntitySpan,
        chunk_range: tuple[int, int] | None,
    ) -> int:
        """
        Calculates how many characters of an entity occur inside a chunk.
        """

        if chunk_range is None:
            return 0

        chunk_start, chunk_end = chunk_range

        """
        Standard interval-overlap formula.
        Suppose:
            Entity: [80, 120)
            Chunk:  [0, 100)

        The overlapping part is:
            [80, 100)

        Its length is 20 characters.
        The calculation is:
            min(120, 100) - max(80, 0)
        which becomes:
            100 - 80
        which is:
            20
        max(0, ...) prevents a negative result when the ranges do not overlap.
        """
        return max(
            0,
            min(span.end, chunk_end)
            - max(span.start, chunk_start),
        )

    def _find_span_owners(
        self,
        spans: list[EntitySpan],
        chunk_ranges: list[
            tuple[int, int] | None
        ],
    ) -> dict[int, int]:
        """
        Assign each entity span to exactly one token chunk.

        This prevents entities in overlapping tokenizer windows
        from appearing more than once in training/evaluation.
        """

        owners: dict[int, int] = {}

        for span_index, span in enumerate(spans):

            """
            Example:
                Chunk:  [0, 100)
                Entity: [30, 50)
            The chunk fully contains the entity.
            But:
                Chunk:  [0, 100)
                Entity: [80, 120)

            does not fully contain it.
            """
            containing_chunks = [
                chunk_index
                for chunk_index, chunk_range
                in enumerate(chunk_ranges)
                if (
                    chunk_range is not None
                    and chunk_range[0] <= span.start
                    and span.end <= chunk_range[1]
                )
            ]

            # If one or more chunks fully contain the entity,
            # assign it to the first one.
            # It might appear completely in two 
            # chunks because of the stride overlap.
            # Choosing the first prevents duplicate training labels.
            if containing_chunks:
                owners[span_index] = (
                    containing_chunks[0]
                )
                continue
            
            # If no chunk completely contains the entity, 
            # calculate how much of it appears in every chunk.
            overlaps = [
                self._calculate_overlap(
                    span,
                    chunk_range,
                )
                for chunk_range in chunk_ranges
            ]

            # no chunks => nowhere to assign the entity.
            if not overlaps:
                continue

            best_chunk = int(np.argmax(overlaps))

            # Only assign the entity 
            # if the selected chunk actually overlaps it.
            if overlaps[best_chunk] > 0:
                owners[span_index] = best_chunk

        return owners

    def _label_chunk(
        self,
        offsets: list[tuple[int, int]],
        chunk_index: int,
        spans: list[EntitySpan],
        span_owners: Mapping[int, int],
    ) -> list[int]:
        """
        This method produces one label ID for every token in one chunk.
        For example:
            [0, 0, 1, 2, -100]
        could mean:
            Token 1 → O
            Token 2 → O
            Token 3 → B-INVOICE_NUMBER
            Token 4 → I-INVOICE_NUMBER
            Token 5 → ignored
        """

        labels: list[int] = []
        started_spans: set[int] = set()

        # Process every token’s character range.
        for token_start, token_end in offsets:
            # Special and padding tokens.
            # A token with an empty range is usually:
            # * `[CLS]`
            # * `[SEP]`
            # * Padding
            if token_start == token_end:
                labels.append(-100)
                continue

            overlapping_span: (
                tuple[int, EntitySpan] | None
            ) = None

            for span_index, span in enumerate(spans):
                # Check whether the token range and entity range overlap.
                overlaps = (
                    token_start < span.end
                    and token_end > span.start
                )
                # Store the first matching entity and stop searching.
                # This is safe because InvoiceSpanAligner
                # prevents entity spans from overlapping each other.
                if overlaps:
                    overlapping_span = (
                        span_index,
                        span,
                    )
                    break
            
            # If the token is not inside an entity
            # give it the `O` label
            if overlapping_span is None:
                labels.append(
                    self.label2id["O"]
                )
                continue

            span_index, span = overlapping_span

            # Ignore duplicate span copies produced by
            # overlapping tokenizer windows.
            # The entity may appear in multiple overlapping chunks.
            # If this chunk is not the entity’s selected owner, ignore its token with -100.
            # This avoids counting or training on the entity twice.
            if span_owners.get(span_index) != chunk_index:
                labels.append(-100)
                continue

            prefix = (
                "B"
                if span_index not in started_spans
                else "I"
            )

            label_name = (
                f"{prefix}-{span.entity}"
            )

            # Check that the label exists in the configured label mapping.
            # This catches situations where you add a new entity to InvoiceTarget
            # but forget to add its BIO labels to label2id.
            if label_name not in self.label2id:
                # Stop with an informative error rather than 
                # silently assigning an incorrect label.
                raise KeyError(
                    f"Missing label in label2id: "
                    f"{label_name}"
                )

            # Convert the label string to a number and 
            # add it to the result.
            labels.append(
                self.label2id[label_name]
            )
            started_spans.add(span_index)

        return labels

    @staticmethod
    def _parse_target(
        value: object,
    ) -> InvoiceTarget:
        """
        Handles three possible input formats: 
        * InvoiceTarget
        * str
        * dict
        """

        if isinstance(value, InvoiceTarget):
            return value

        if isinstance(value, str):
            return InvoiceTarget.model_validate_json(
                value
            )

        return InvoiceTarget.model_validate(
            value
        )

    def __call__(
        self,
        examples: Mapping[str, list],
    ) -> TokenizedBatch:
        """
        Makes the object callable for passing it to Hugging Face
        """
        
        """
        The tokenizer declares the inputs expected by its model.
        For BERT, these are commonly:
            [
                "input_ids",
                "token_type_ids",
                "attention_mask",
            ]
        The code creates:
            {
                "input_ids": [],
                "token_type_ids": [],
                "attention_mask": [],
            }
        """
        output: TokenizedBatch = {
            "input_ids": [],
            "token_type_ids": [],
            "attention_mask": [],
            "labels": []
        }

        # Processing every document in the batch
        for text, json_data in zip(
            examples["ocred_text"],
            examples["json_data"],
        ):
            text = str(text)

            target = self._parse_target(
                value=json_data
            )

            # Finding entity character spans
            # Connects the target JSON values with 
            # their locations in the OCR text.
            # At this point, labels exist at the character level
            # They are not yet token labels.
            spans = self.aligner.find_spans(
                text=text,
                target=target,
            )
            
            encoding = self.tokenizer(
                # Pass the OCR text into the Hugging Face tokenizer
                text,
                # The tokenizer must respect `max_length`.
                # Because overflow handling is enabled, 
                # text beyond the first chunk is placed into additional 
                # chunks rather than simply discarded.
                truncation=True,
                # Each chunk can contain at most this many tokens.
                max_length=self.max_length,
                # Neighbouring chunks overlap by this many tokens.
                stride=self.window_stride,
                # Create multiple chunks when the document is too long.
                # Without this, the text beyond `max_length` would be lost.
                return_overflowing_tokens=True,
                # Return each token’s original character location.
                # For example:
                #   "12345"
                # might become:
                #   "123"   → (16, 19)
                #   "##45"  → (19, 21)
                # These offsets are what connect character spans 
                # from `InvoiceSpanAligner` to BERT tokens.
                return_offsets_mapping=True,
            )

            # Remove offsets from `encoding` and save them separately.
            # Offsets are needed to build labels,
            # but they are not an input to BERT.
            offset_chunks = encoding.pop(
                "offset_mapping"
            )

            # The tokenizer can return metadata showing which original example produced each overflow chunk.
            # This encoder processes one document at a time inside the loop, so that metadata is unnecessary.
            # The second argument, `None`, means “do not raise an error if this key does not exist.”
            encoding.pop(
                "overflow_to_sample_mapping",
                None,
            )

            # Finding chunk character ranges
            # For each chunk, calculate the portion of the original text it covers.
            # Example:
            #   [
            #       (0, 1800),
            #       (1550, 3200),
            #   ]
            # The overlap from characters 1550–1800 comes from the tokenizer stride.
            chunk_ranges = [
                self._get_chunk_range(offsets)
                for offsets in offset_chunks
            ]

            # Selecting one owner per entity
            # Choose exactly one chunk responsible for each entity.
            # Example:
            #   {
            #       0: 0,
            #       1: 0,
            #       2: 1,
            #   }
            # This prevents duplicate entity labels in overlapping chunks.
            span_owners = self._find_span_owners(
                spans=spans,
                chunk_ranges=chunk_ranges,
            )

            # Creating labels for each chunk
            # Process every chunk created from this document.
            for chunk_index, offsets in enumerate(
                offset_chunks
            ):
                # Create one numerical label per token in this chunk.
                # Example:
                #   [-100, 0, 0, 1, 2, -100]
                labels = self._label_chunk(
                    offsets=offsets,
                    chunk_index=chunk_index,
                    spans=spans,
                    span_owners=span_owners,
                )

                # Saving model inputs
                # Process each input required by BERT:
                # * input_ids
                # * attention_mask
                # * token_type_ids

                for name in (
                    self.tokenizer.model_input_names
                ):
                    # Take the input values for the current chunk and 
                    # add them to the output batch.
                    output[name].append(
                        encoding[name][chunk_index]
                    )

                # Add the labels belonging to the same chunk.
                # The positions line up:
                #   input_ids[0][5]
                # and:
                #   labels[0][5]
                # refer to the same token.

                output["labels"].append(labels)

        # Return all generated chunks and labels.
        # The result looks approximately like:
        # {
        #     "input_ids": [
        #         [101, 1999, 6764, 102, ...],
        #         [101, 2023, 2003, 102, ...],
        #     ],
        #     "attention_mask": [
        #         [1, 1, 1, 1, ...],
        #         [1, 1, 1, 1, ...],
        #     ],
        #     "token_type_ids": [
        #         [0, 0, 0, 0, ...],
        #         [0, 0, 0, 0, ...],
        #     ],
        #     "labels": [
        #         [-100, 0, 1, -100, ...],
        #         [-100, 0, 0, -100, ...],
        #     ],
        # }
        # Each row represents one chunk, not necessarily one original invoice.
        # One long invoice can produce several rows.
        return output

#### Test

In [83]:
row = df.iloc[12]
text = row['ocred_text']
text

'Invoice no: 32135544 Date of issue: 08/16/2014 Seller: Client: Reed and Sons Lucas-Barnett Unit 5010 Box 0461 USNS Scott DPO AA 27717 FPO AA 73490 Tax Id: 912-95-0493 Tax Id: 997-85-1616 IBAN: GB2IVGUQ93040788957754 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth Wine Bottle Storage 6 Slots 4,00 each 30,59 122,36 10% 134,60 Modern Countertop Metal Rack Kitchen Bar Organizer 2 Anthropologie Gold Elegant 4,00 each 19,98 79,92 10% 87,91 Swan Decorative Metal Bottle Stopper Wine Saver 3 mDesign Metal Wine Rack 4,00 each 44,99 179,96 10% 197,96 Storage Organizer, 12 Bottles Dark Gray Tiffany and Co Pair 2 Stemless 2,00 each 74,99 149,98 10% 164,98 Wine Champagne Glasses Original Stickers NO BOX 5 Outdoor Wine Glass Beer Holder 3,00 each 11,44 34,32 10% 37,75 To Armrest With Attachments Red Wine Base _yoxq Hardwood Wine Rack 12 Bottles 3,00 each 20,00 60,00 10% 66,00 Very Nice Eonophiles Delight SUMMARY VAT [%] Net worth VAT Gross worth 10% 626,54 62,65 689,19 Total $ 6

##### Arrange the real encoder input

The active label schema contains the eight baseline entities. The invoice IBAN remains in the OCR text, but it is intentionally omitted from this target until `ACCOUNT_NUMBER` is added to `ENTITY_TYPES`.

In [99]:
tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased",
    use_fast=True,
)

target = InvoiceTarget(
    invoice=InvoiceData(
        invoice_number="32135544",
        invoice_date="08/16/2014",
        seller_name="Reed and Sons",
        seller_address=(
            "Unit 5010 Box 0461\n"
            "DPO AA 27717"
        ),
        client_name="Lucas-Barnett",
        client_address=(
            "USNS Scott\n"
            "FPO AA 73490"
        ),
    ),
    subtotal=SubtotalData(
        tax="62.65",
        total="689.19",
    ),
    payment_instructions=(
        PaymentInstructions()
    ),
    items=[],
)

batch_encoder = TokenLabelBatchEncoder(
    tokenizer=tokenizer,
    aligner=document_aligner,
    label2id=label2id,
    max_length=512,
    window_stride=64,
)

examples = {
    "ocred_text": [text],
    "json_data": [target],
}

encoded = batch_encoder(examples)

encoded = batch_encoder(examples)

full_encoding = tokenizer(
    text,
    truncation=False,
    add_special_tokens=True,
)

full_token_count = len(
    full_encoding["input_ids"]
)

print("Full token count:", full_token_count)
print("Maximum length:", batch_encoder.max_length)

Full token count: 328
Maximum length: 512


##### Test 1: output structure

Do not assert that multiple chunks were produced.

In [100]:
required_keys = {
    "input_ids",
    "attention_mask",
    "labels",
}

if (
    "token_type_ids"
    in tokenizer.model_input_names
):
    required_keys.add("token_type_ids")

assert required_keys.issubset(encoded), (
    f"Missing output keys: "
    f"{required_keys - set(encoded)}"
)

chunk_count = len(encoded["input_ids"])

assert chunk_count >= 1

if full_token_count <= batch_encoder.max_length:
    assert chunk_count == 1, (
        f"The invoice contains {full_token_count} "
        "tokens and should fit into one chunk, "
        f"but {chunk_count} chunks were produced."
    )

for name in required_keys:
    assert len(encoded[name]) == chunk_count

print(
    "Test 1 passed:",
    chunk_count,
    "chunk(s) produced.",
)


Test 1 passed: 1 chunk(s) produced.


##### Test 2: full document coverage

In [101]:
coverage_encoding = tokenizer(
    text,
    truncation=True,
    max_length=batch_encoder.max_length,
    stride=batch_encoder.window_stride,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

offset_chunks = coverage_encoding[
    "offset_mapping"
]

chunk_ranges = [
    batch_encoder._get_chunk_range(offsets)
    for offsets in offset_chunks
]

valid_chunk_ranges = [
    chunk_range
    for chunk_range in chunk_ranges
    if chunk_range is not None
]

assert valid_chunk_ranges

covered_start = min(
    start
    for start, end in valid_chunk_ranges
)

covered_end = max(
    end
    for start, end in valid_chunk_ranges
)

expected_end = len(text.rstrip())

assert covered_start == 0

assert covered_end >= expected_end, (
    f"Tokenizer coverage ends at character "
    f"{covered_end}, but the document ends at "
    f"{expected_end}."
)

print(
    "Test 2 passed:",
    f"characters [{covered_start}, {covered_end}) "
    "are covered.",
)

Test 2 passed: characters [0, 1021) are covered.


##### Test 3: aligned sequence lengths

Use the encoder configuration rather than a hardcoded 128.

In [102]:
for chunk_index in range(chunk_count):
    sequence_length = len(
        encoded["input_ids"][chunk_index]
    )

    assert sequence_length > 0

    assert (
        sequence_length
        <= batch_encoder.max_length
    )

    for name in required_keys:
        actual_length = len(
            encoded[name][chunk_index]
        )

        assert actual_length == sequence_length, (
            f"Chunk {chunk_index}: {name} has "
            f"length {actual_length}; expected "
            f"{sequence_length}."
        )

print(
    "Test 3 passed: every chunk has aligned "
    "input and label lengths."
)

Test 3 passed: every chunk has aligned input and label lengths.


##### Test 4: special-token labels

In [103]:
for chunk_index, labels in enumerate(
    encoded["labels"]
):
    assert labels[0] == -100, (
        f"Chunk {chunk_index}: first special "
        "token was not ignored."
    )

    assert labels[-1] == -100, (
        f"Chunk {chunk_index}: final special "
        "token was not ignored."
    )

print(
    "Test 4 passed: special tokens use -100."
)

Test 4 passed: special tokens use -100.


##### Test 5: expected entity counts

In [104]:
generated_labels = [
    id2label[label_id]
    for chunk_labels in encoded["labels"]
    for label_id in chunk_labels
    if label_id != -100
]

begin_counts = Counter(
    label
    for label in generated_labels
    if label.startswith("B-")
)

expected_begin_counts = Counter({
    "B-INVOICE_NUMBER": 1,
    "B-INVOICE_DATE": 1,
    "B-SELLER_NAME": 1,
    "B-SELLER_ADDRESS": 2,
    "B-CLIENT_NAME": 1,
    "B-CLIENT_ADDRESS": 2,
    "B-TAX": 1,
    "B-TOTAL": 1,
})

assert begin_counts == expected_begin_counts, (
    f"\nExpected: {expected_begin_counts}"
    f"\nActual:   {begin_counts}"
)

print(
    "Test 5 passed: expected entity spans "
    "were labeled exactly once."
)

begin_counts

Test 5 passed: expected entity spans were labeled exactly once.


Counter({'B-INVOICE_NUMBER': 1,
         'B-INVOICE_DATE': 1,
         'B-SELLER_NAME': 1,
         'B-CLIENT_NAME': 1,
         'B-SELLER_ADDRESS': 2,
         'B-CLIENT_ADDRESS': 2,
         'B-TAX': 1,
         'B-TOTAL': 1})

##### Test 6: valid label IDs

In [105]:
valid_label_ids = set(label2id.values())

invalid_label_ids = [
    label_id
    for chunk_labels in encoded["labels"]
    for label_id in chunk_labels
    if (
        label_id != -100
        and label_id not in valid_label_ids
    )
]

assert not invalid_label_ids, (
    f"Invalid label IDs: {invalid_label_ids}"
)

assert label2id["O"] in {
    label_id
    for chunk_labels in encoded["labels"]
    for label_id in chunk_labels
}

print(
    "Test 6 passed: all label IDs are valid."
)

Test 6 passed: all label IDs are valid.


##### Test 7: valid BIO sequences

In [106]:
for chunk_index, chunk_labels in enumerate(
    encoded["labels"]
):
    previous_label = None

    for label_id in chunk_labels:
        if label_id == -100:
            previous_label = None
            continue

        label = id2label[label_id]

        if label == "O":
            previous_label = None
            continue

        prefix, entity = label.split(
            "-",
            maxsplit=1,
        )

        assert prefix in {"B", "I"}

        if prefix == "I":
            valid_previous_labels = {
                f"B-{entity}",
                f"I-{entity}",
            }

            assert (
                previous_label
                in valid_previous_labels
            ), (
                f"Chunk {chunk_index}: {label} "
                f"incorrectly follows "
                f"{previous_label}."
            )

        previous_label = label

print(
    "Test 7 passed: BIO sequences are valid."
)

Test 7 passed: BIO sequences are valid.


##### Test 8: alignment spans correspond to BIO beginnings

This connects DocumentAligner with TokenLabelBatchEncoder.

In [107]:
spans = document_aligner.find_spans(
    text=text,
    target=target,
)

expected_span_counts = Counter(
    f"B-{span.entity}"
    for span in spans
)

assert begin_counts == expected_span_counts, (
    "The number of B labels does not match "
    "the number of aligned character spans."
    f"\nAligned spans: {expected_span_counts}"
    f"\nBIO labels:    {begin_counts}"
)

print(
    "Test 8 passed: every aligned character "
    "span produced one B label."
)

Test 8 passed: every aligned character span produced one B label.


#####  Test 9: supported target formats

In [108]:
encoded_from_json = batch_encoder({
    "ocred_text": [text],
    "json_data": [
        target.model_dump_json()
    ],
})

encoded_from_dict = batch_encoder({
    "ocred_text": [text],
    "json_data": [
        target.model_dump()
    ],
})

assert encoded_from_json == encoded
assert encoded_from_dict == encoded

print(
    "Test 9 passed: InvoiceTarget, JSON string, "
    "and dictionary targets produce identical output."
)

Test 9 passed: InvoiceTarget, JSON string, and dictionary targets produce identical output.


##### Test 10: invalid configuration

In [109]:
invalid_configurations = [
    {
        "max_length": 0,
        "window_stride": 0,
        "labels": label2id,
    },
    {
        "max_length": 512,
        "window_stride": 512,
        "labels": label2id,
    },
    {
        "max_length": 512,
        "window_stride": 64,
        "labels": {
            "B-INVOICE_NUMBER": 0,
        },
    },
]

for configuration in invalid_configurations:
    try:
        TokenLabelBatchEncoder(
            tokenizer=tokenizer,
            aligner=document_aligner,
            label2id=configuration["labels"],
            max_length=configuration[
                "max_length"
            ],
            window_stride=configuration[
                "window_stride"
            ],
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Expected configuration to fail: "
            f"{configuration}"
        )

print(
    "Test 10 passed: invalid configurations "
    "are rejected."
)

Test 10 passed: invalid configurations are rejected.


##### Inspect the entity tokens produced by the encoder

In [110]:
token_label_rows = []

for chunk_index, (
    input_ids,
    label_ids,
) in enumerate(zip(
    encoded["input_ids"],
    encoded["labels"],
)):
    tokens = tokenizer.convert_ids_to_tokens(
        input_ids
    )

    for token_index, (
        token,
        label_id,
    ) in enumerate(zip(tokens, label_ids)):
        if label_id == -100:
            continue

        label = id2label[label_id]

        if label == "O":
            continue

        token_label_rows.append({
            "chunk": chunk_index,
            "token_index": token_index,
            "token": token,
            "label": label,
        })

token_labels_df = pd.DataFrame(
    token_label_rows
)

token_labels_df

,chunk,token_index,token,label
0,0,6,321,B-INVOICE_NUMBER
1,0,7,##35,I-INVOICE_NUMBER
2,0,8,##54,I-INVOICE_NUMBER
3,0,9,##4,I-INVOICE_NUMBER
4,0,14,08,B-INVOICE_DATE
5,0,15,/,I-INVOICE_DATE
6,0,16,16,I-INVOICE_DATE
7,0,17,/,I-INVOICE_DATE
8,0,18,2014,I-INVOICE_DATE
9,0,23,reed,B-SELLER_NAME


### Hugging Face Dataset Builder

In [111]:
from datasets import Dataset, DatasetDict

@dataclass(frozen=True, slots=True)
class DatasetBundle:
    raw: DatasetDict
    tokenized: DatasetDict


@dataclass(slots=True)
class HuggingFaceDatasetBuilder:
    batch_encoder: TokenLabelBatchEncoder
    batch_size: int = 16
    text_column: str = "ocred_text"
    target_column: str = "json_data"

    def _from_dataframe(
        self,
        frame: pd.DataFrame,
    ) -> Dataset:
        return Dataset.from_pandas(
            frame[
                [
                    self.text_column,
                    self.target_column,
                ]
            ],
            preserve_index=False,
        )

    def build(
        self,
        *,
        train: pd.DataFrame,
        validation: pd.DataFrame,
        test: pd.DataFrame,
    ) -> DatasetBundle:
        raw = DatasetDict({
            "train": self._from_dataframe(
                train
            ),
            "validation": self._from_dataframe(
                validation
            ),
            "test": self._from_dataframe(
                test
            ),
        })

        tokenized = raw.map(
            self.batch_encoder,
            batched=True,
            batch_size=self.batch_size,
            remove_columns=[
                self.text_column,
                self.target_column,
            ],
            load_from_cache_file=False,
        )

        return DatasetBundle(
            raw=raw,
            tokenized=tokenized,
        )

#### Test

In [112]:
row = df.iloc[13]
text = row['ocred_text']
text

"Invoice no: 67117171 Date of issue: 01/27/2021 Seller: Client: Gilmore, Brady and Carter Phillips-Moreno 56436 Dam Suite 437 1293 Lawrence Estates Suite 940 Lake Rebecca, LA 78907 Besthaven, NV 23717 Tax Id: 905-96-1343 Tax Id: 953-91-5577 IBAN: GB86OULB80821848504333 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth The Complete Idiots Guide to 4,00 each 4,49 17,96 10% 19,76 Trouble-Free Home Re 2 Aesop's Fables Aesop Illustrated 4,00 each 34,65 138,60 10% 152,46 By A. Rackham Sealed Leather Bound Collectible SUMMARY VAT [%] Net worth VAT Gross worth 10% 156,56 15,66 172,22 Total $ 156,56 $ 15,66 $ 172,22 Amy"

In [134]:
dataset_batch_encoder = TokenLabelBatchEncoder(
    tokenizer=tokenizer,
    aligner=document_aligner,
    label2id=label2id,
    max_length=512,
    window_stride=64,
)

target =  InvoiceTarget(
    invoice=InvoiceData(
        invoice_number="67117171",
        invoice_date="01/27/2021",
        seller_name=(
            "Gilmore, Brady and Carter"
        ),
        seller_address=(
            "56436 Dam Suite 437\n"
            "Lake Rebecca, LA 78907"
        ),
        client_name="Phillips-Moreno",
        client_address=(
            "1293 Lawrence Estates Suite 940\n"
            "Besthaven, NV 23717"
        ),
    ),
    subtotal=SubtotalData(
        tax="15.66",
        total="172.22",
    ),
    # ACCOUNT_NUMBER is not currently part
    # of the baseline BIO label schema.
    payment_instructions=(
        PaymentInstructions()
    ),
    items=[],
)

test_record = {
    "ocred_text": text,
    "json_data": target.model_dump_json(),
}

builder_train_df = pd.DataFrame([
    {
        **test_record,
        "source_id": "train-1",
    },
    {
        **test_record,
        "source_id": "train-2",
    },
])

builder_validation_df = pd.DataFrame([
    {
        **test_record,
        "source_id": "validation-1",
    },
])

builder_test_df = pd.DataFrame([
    {
        **test_record,
        "source_id": "test-1",
    },
])

dataset_builder = HuggingFaceDatasetBuilder(
    batch_encoder=dataset_batch_encoder,
    batch_size=2,
)

dataset_bundle = dataset_builder.build(
    train=builder_train_df,
    validation=builder_validation_df,
    test=builder_test_df,
)

/usr/local/lib/python3.13/dist-packages/dill/_dill.py:423: PicklingWarning: Cannot locate reference to <enum 'EntityType'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.13/dist-packages/dill/_dill.py:423: PicklingWarning: Cannot pickle <enum 'EntityType'>: __main__.EntityType has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.13/dist-packages/dill/_dill.py:423: PicklingWarning: Cannot locate reference to <class '__main__.InvoiceData'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.13/dist-packages/dill/_dill.py:423: PicklingWarning: Cannot pickle <class '__main__.InvoiceData'>: __main__.InvoiceData has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

##### Test 1: result contains raw and tokenized datasets

In [135]:
assert isinstance(
    dataset_bundle,
    DatasetBundle,
)

assert isinstance(
    dataset_bundle.raw,
    DatasetDict,
)

assert isinstance(
    dataset_bundle.tokenized,
    DatasetDict,
)

print(
    "Test 1 passed: DatasetBundle contains "
    "raw and tokenized DatasetDict objects."
)

Test 1 passed: DatasetBundle contains raw and tokenized DatasetDict objects.


##### Test 2: expected splits exist

In [136]:
expected_splits = {
    "train",
    "validation",
    "test",
}

assert (
    set(dataset_bundle.raw)
    == expected_splits
)

assert (
    set(dataset_bundle.tokenized)
    == expected_splits
)

print(
    "Test 2 passed: train, validation, "
    "and test splits exist."
)

Test 2 passed: train, validation, and test splits exist.


##### Test 3: raw document counts are preserved


In [137]:
expected_document_counts = {
    "train": 2,
    "validation": 1,
    "test": 1,
}

actual_document_counts = {
    split: len(dataset_bundle.raw[split])
    for split in expected_splits
}

assert (
    actual_document_counts
    == expected_document_counts
), (
    f"Expected {expected_document_counts}, "
    f"received {actual_document_counts}"
)

print(
    "Test 3 passed: raw document counts "
    "were preserved."
)

Test 3 passed: raw document counts were preserved.


##### Test 4: only required raw columns are retained

The additional source_id column should not enter the Hugging Face dataset.


In [138]:
expected_raw_columns = {
    "ocred_text",
    "json_data",
}

for split in expected_splits:
    actual_columns = set(
        dataset_bundle.raw[
            split
        ].column_names
    )

    assert actual_columns == (
        expected_raw_columns
    )

    assert "source_id" not in actual_columns
    assert "__index_level_0__" not in (
        actual_columns
    )

print(
    "Test 4 passed: only OCR text and "
    "JSON target columns were retained."
)

Test 4 passed: only OCR text and JSON target columns were retained.


##### Test 5: raw values and ordering are preserved

In [139]:
source_frames = {
    "train": builder_train_df,
    "validation": builder_validation_df,
    "test": builder_test_df,
}

for split, source_frame in (
    source_frames.items()
):
    raw_dataset = dataset_bundle.raw[split]

    assert list(
        raw_dataset["ocred_text"]
    ) == source_frame[
        "ocred_text"
    ].tolist()

    assert list(
        raw_dataset["json_data"]
    ) == source_frame[
        "json_data"
    ].tolist()

print(
    "Test 5 passed: raw values and row "
    "ordering were preserved."
)

Test 5 passed: raw values and row ordering were preserved.


##### Test 6: tokenized datasets contain model inputs

In [140]:
required_tokenized_columns = {
    "input_ids",
    "attention_mask",
    "labels",
}

if (
    "token_type_ids"
    in tokenizer.model_input_names
):
    required_tokenized_columns.add(
        "token_type_ids"
    )

for split in expected_splits:
    actual_columns = set(
        dataset_bundle.tokenized[
            split
        ].column_names
    )

    assert (
        required_tokenized_columns
        .issubset(actual_columns)
    )

    assert "ocred_text" not in (
        actual_columns
    )

    assert "json_data" not in (
        actual_columns
    )

print(
    "Test 6 passed: tokenized datasets "
    "contain model inputs and labels."
)

Test 6 passed: tokenized datasets contain model inputs and labels.


##### Test 7: every source document produces one chunk

This assertion is valid for the current invoice because it fits within 512 tokens.

In [141]:
assert (
    full_token_count
    <= batch_encoder.max_length
)

for split in expected_splits:
    document_count = len(
        dataset_bundle.raw[split]
    )

    chunk_count = len(
        dataset_bundle.tokenized[split]
    )

    assert chunk_count == document_count, (
        f"{split}: expected {document_count} "
        f"chunks, received {chunk_count}."
    )

print(
    "Test 7 passed: every test document "
    "produced exactly one chunk."
)

Test 7 passed: every test document produced exactly one chunk.


For the full dataset, do not generally assume one document equals one chunk. Longer documents can create multiple chunks.

##### Test 8: every tokenized row has consistent lengths


In [142]:
for split in expected_splits:
    for chunk_index, row in enumerate(
        dataset_bundle.tokenized[split]
    ):
        sequence_length = len(
            row["input_ids"]
        )

        assert sequence_length > 0

        assert (
            sequence_length
            <= batch_encoder.max_length
        )

        assert len(
            row["attention_mask"]
        ) == sequence_length

        assert len(
            row["labels"]
        ) == sequence_length

        if "token_type_ids" in row:
            assert len(
                row["token_type_ids"]
            ) == sequence_length

print(
    "Test 8 passed: every tokenized row "
    "has consistent sequence lengths."
)


Test 8 passed: every tokenized row has consistent sequence lengths.


##### Test 9: all generated label IDs are valid

In [143]:
valid_label_ids = set(
    label2id.values()
)

for split in expected_splits:
    for chunk_index, row in enumerate(
        dataset_bundle.tokenized[split]
    ):
        invalid_label_ids = [
            label_id
            for label_id in row["labels"]
            if (
                label_id != -100
                and label_id
                not in valid_label_ids
            )
        ]

        assert not invalid_label_ids, (
            f"{split}, chunk {chunk_index}: "
            f"invalid labels "
            f"{invalid_label_ids}"
        )

print(
    "Test 9 passed: all generated label "
    "IDs are valid."
)


Test 9 passed: all generated label IDs are valid.


##### Test 10: special tokens are ignored

In [144]:
for split in expected_splits:
    for chunk_index, row in enumerate(
        dataset_bundle.tokenized[split]
    ):
        labels = row["labels"]

        assert labels[0] == -100, (
            f"{split}, chunk {chunk_index}: "
            "first special token is not ignored."
        )

        assert labels[-1] == -100, (
            f"{split}, chunk {chunk_index}: "
            "last special token is not ignored."
        )

print(
    "Test 10 passed: special tokens "
    "are excluded from the loss."
)

Test 10 passed: special tokens are excluded from the loss.


#####  Test 11: builder output equals direct encoder output

This is the main functional test. It verifies that the builder passes the correct values to `TokenLabelBatchEncoder`.

In [145]:
for split, source_frame in (
    source_frames.items()
):
    direct_output = batch_encoder({
        "ocred_text": source_frame[
            "ocred_text"
        ].tolist(),
        "json_data": source_frame[
            "json_data"
        ].tolist(),
    })

    built_dataset = (
        dataset_bundle.tokenized[split]
    )

    for name, expected_values in (
        direct_output.items()
    ):
        actual_values = list(
            built_dataset[name]
        )

        assert actual_values == (
            expected_values
        ), (
            f"{split}: builder output for "
            f"{name} differs from direct "
            "encoder output."
        )

print(
    "Test 11 passed: builder output matches "
    "direct TokenLabelBatchEncoder output."
)

Test 11 passed: builder output matches direct TokenLabelBatchEncoder output.


##### Test 12: each document contains expected entities

In [147]:
expected_per_document = Counter({
    "B-INVOICE_NUMBER": 1,
    "B-INVOICE_DATE": 1,
    "B-SELLER_NAME": 1,
    "B-SELLER_ADDRESS": 2,
    "B-CLIENT_NAME": 1,
    "B-CLIENT_ADDRESS": 2,
    "B-TAX": 1,
    "B-TOTAL": 1,
})

for split in expected_splits:
    document_count = len(
        dataset_bundle.raw[split]
    )

    actual_begin_counts = Counter(
        id2label[label_id]
        for row in (
            dataset_bundle
            .tokenized[split]
        )
        for label_id in row["labels"]
        if (
            label_id != -100
            and id2label[label_id]
            .startswith("B-")
        )
    )

    expected_begin_counts = Counter({
        label: count * document_count
        for label, count
        in expected_per_document.items()
    })

    assert actual_begin_counts == (
        expected_begin_counts
    ), (
        f"{split}:"
        f"\nExpected: {expected_begin_counts}"
        f"\nActual:   {actual_begin_counts}"
    )

print(
    "Test 12 passed: all documents contain "
    "the expected BIO entities."
)

Test 12 passed: all documents contain the expected BIO entities.


##### Test 13: source DataFrames were not modified

In [148]:
assert set(
    builder_train_df.columns
) == {
    "ocred_text",
    "json_data",
    "source_id",
}

assert set(
    builder_validation_df.columns
) == {
    "ocred_text",
    "json_data",
    "source_id",
}

assert set(
    builder_test_df.columns
) == {
    "ocred_text",
    "json_data",
    "source_id",
}

print(
    "Test 13 passed: source DataFrames "
    "were not modified."
)

Test 13 passed: source DataFrames were not modified.


##### Test 14: missing required columns fail clearly


In [149]:
invalid_train_df = (
    builder_train_df
    .drop(columns=["json_data"])
)

try:
    dataset_builder.build(
        train=invalid_train_df,
        validation=builder_validation_df,
        test=builder_test_df,
    )
except KeyError as error:
    assert "json_data" in str(error)
else:
    raise AssertionError(
        "Expected a KeyError when json_data "
        "is missing."
    )

print(
    "Test 14 passed: missing required "
    "columns are rejected."
)

Test 14 passed: missing required columns are rejected.


##### Display the result

In [151]:
dataset_summary = pd.DataFrame([
    {
        "split": split,
        "documents": len(
            dataset_bundle.raw[split]
        ),
        "tokenized_chunks": len(
            dataset_bundle.tokenized[split]
        ),
        "columns": (
            dataset_bundle
            .tokenized[split]
            .column_names
        ),
    }
    for split in [
        "train",
        "validation",
        "test",
    ]
])

dataset_summary

,split,documents,tokenized_chunks,columns
0,train,2,2,"[input_ids, token_type_ids, attention_mask, la..."
1,validation,1,1,"[input_ids, token_type_ids, attention_mask, la..."
2,test,1,1,"[input_ids, token_type_ids, attention_mask, la..."


### Model metrics

In [153]:
%pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 8.2 MB/s eta 0:00:00a 0:00:01


In [154]:
from collections.abc import Mapping
from dataclasses import dataclass

import numpy as np
import torch
from seqeval.metrics import (
    f1_score,
    precision_score,
    recall_score,
)
from torchmetrics.functional.classification import (
    multiclass_accuracy,
    multiclass_f1_score,
)
from transformers import EvalPrediction

In [155]:
@dataclass(frozen=True, slots=True)
class TokenClassificationMetrics:
    id2label: Mapping[int, str]

    @property
    def number_of_labels(self) -> int:
        return len(self.id2label)

    def decode(
        self,
        predictions: np.ndarray,
        label_ids: np.ndarray,
    ) -> tuple[
        list[list[str]],
        list[list[str]],
    ]:
        predicted_ids = np.argmax(
            predictions,
            axis=-1,
        )

        predicted_labels: list[
            list[str]
        ] = []

        actual_labels: list[
            list[str]
        ] = []

        for predicted_sequence, actual_sequence in zip(
            predicted_ids,
            label_ids,
        ):
            sequence_predictions = []
            sequence_actual = []

            for predicted_id, actual_id in zip(
                predicted_sequence,
                actual_sequence,
            ):
                if actual_id == -100:
                    continue

                sequence_predictions.append(
                    self.id2label[
                        int(predicted_id)
                    ]
                )

                sequence_actual.append(
                    self.id2label[
                        int(actual_id)
                    ]
                )

            predicted_labels.append(
                sequence_predictions
            )

            actual_labels.append(
                sequence_actual
            )

        return (
            predicted_labels,
            actual_labels,
        )

    def __call__(
        self,
        evaluation: EvalPrediction,
    ) -> dict[str, float]:
        predictions = evaluation.predictions

        # Some models return a tuple where logits are
        # the first item.
        if isinstance(predictions, tuple):
            predictions = predictions[0]

        label_ids = evaluation.label_ids

        predicted_ids = np.argmax(
            predictions,
            axis=-1,
        )

        valid_mask = label_ids != -100

        token_predictions = torch.as_tensor(
            predicted_ids[valid_mask],
            dtype=torch.long,
        )

        token_targets = torch.as_tensor(
            label_ids[valid_mask],
            dtype=torch.long,
        )

        predicted_labels, actual_labels = (
            self.decode(
                predictions=predictions,
                label_ids=label_ids,
            )
        )

        return {
            # Entity-level BIO metrics.
            "precision": precision_score(
                actual_labels,
                predicted_labels,
                zero_division=0,
            ),
            "recall": recall_score(
                actual_labels,
                predicted_labels,
                zero_division=0,
            ),
            "f1": f1_score(
                actual_labels,
                predicted_labels,
                zero_division=0,
            ),

            # Token-level TorchMetrics.
            "token_accuracy": float(
                multiclass_accuracy(
                    token_predictions,
                    token_targets,
                    num_classes=(
                        self.number_of_labels
                    ),
                    average="micro",
                ).item()
            ),
            "token_macro_f1": float(
                multiclass_f1_score(
                    token_predictions,
                    token_targets,
                    num_classes=(
                        self.number_of_labels
                    ),
                    average="macro",
                ).item()
            ),
        }

#### Test

##### Setup


In [156]:
metrics = TokenClassificationMetrics(
    id2label=id2label,
)

number_of_labels = len(id2label)

assert metrics.number_of_labels == (
    number_of_labels
)

print("Number of labels:", number_of_labels)

Number of labels: 17


Create a balanced BIO sequence containing every configured label once:

In [158]:
ordered_label_ids = sorted(id2label)

metrics_test_label_ids = np.array([
    [
        -100,                  # [CLS]
        *ordered_label_ids,    # every label
        -100,                  # [SEP]
    ]
], dtype=np.int64)

expected_decoded_labels = [
    id2label[label_id]
    for label_id in ordered_label_ids
]

Helper for turning predicted class IDs into logits:

In [159]:
def create_test_logits(
    predicted_ids: np.ndarray,
    *,
    number_of_labels: int,
    ignored_prediction: int = 0,
) -> np.ndarray:
    safe_predictions = np.where(
        predicted_ids == -100,
        ignored_prediction,
        predicted_ids,
    )

    logits = np.full(
        (
            *predicted_ids.shape,
            number_of_labels,
        ),
        fill_value=-10.0,
        dtype=np.float32,
    )

    np.put_along_axis(
        logits,
        safe_predictions[..., None],
        values=10.0,
        axis=-1,
    )

    return logits

Create perfect predictions

In [160]:
perfect_logits = create_test_logits(
    metrics_test_label_ids,
    number_of_labels=number_of_labels,
    ignored_prediction=label2id["O"],
)

perfect_evaluation = EvalPrediction(
    predictions=perfect_logits,
    label_ids=metrics_test_label_ids,
)

##### Test 1: number of labels

In [161]:
assert metrics.number_of_labels == len(
    id2label
)

assert metrics.number_of_labels == len(
    label2id
)

print(
    "Test 1 passed: number_of_labels "
    "matches the configured label mappings."
)


Test 1 passed: number_of_labels matches the configured label mappings.


#####  Test 2: decode returns label names

In [162]:
(
    predicted_labels,
    actual_labels,
) = metrics.decode(
    predictions=perfect_logits,
    label_ids=metrics_test_label_ids,
)

assert predicted_labels == [
    expected_decoded_labels
]

assert actual_labels == [
    expected_decoded_labels
]

print(
    "Test 2 passed: numerical predictions "
    "were decoded into BIO label names."
)

Test 2 passed: numerical predictions were decoded into BIO label names.


#####  Test 3: ignored tokens are removed

The two -100 positions must not appear in the decoded output.

In [163]:
valid_token_count = int(
    (
        metrics_test_label_ids
        != -100
    ).sum()
)

assert len(predicted_labels) == 1
assert len(actual_labels) == 1

assert len(
    predicted_labels[0]
) == valid_token_count

assert len(
    actual_labels[0]
) == valid_token_count

assert "-100" not in predicted_labels[0]
assert "-100" not in actual_labels[0]

print(
    "Test 3 passed: ignored tokens were "
    "excluded from decoded sequences."
)

Test 3 passed: ignored tokens were excluded from decoded sequences.


##### Test 4: perfect predictions return perfect metrics

In [164]:
perfect_results = metrics(
    perfect_evaluation
)

expected_metric_names = {
    "precision",
    "recall",
    "f1",
    "token_accuracy",
    "token_macro_f1",
}

assert set(perfect_results) == (
    expected_metric_names
)

for metric_name, metric_value in (
    perfect_results.items()
):
    assert isinstance(metric_value, float)

    assert np.isclose(
        metric_value,
        1.0,
    ), (
        f"{metric_name} should be 1.0, "
        f"received {metric_value}."
    )

print(
    "Test 4 passed: perfect predictions "
    "produce perfect metrics."
)

Test 4 passed: perfect predictions produce perfect metrics.


In [165]:
perfect_results

{'precision': np.float64(1.0),
 'recall': np.float64(1.0),
 'f1': np.float64(1.0),
 'token_accuracy': 1.0,
 'token_macro_f1': 1.0}

Because the synthetic sequence includes every configured label, token_macro_f1 should also be exactly 1.0.

##### Test 5: predictions at ignored positions do not affect metrics

Make deliberately incorrect predictions at [CLS] and [SEP].


In [166]:
ignored_positions_logits = (
    perfect_logits.copy()
)

ignored_positions_logits[
    0,
    0,
    :,
] = -10.0

ignored_positions_logits[
    0,
    0,
    label2id["B-INVOICE_NUMBER"],
] = 100.0

ignored_positions_logits[
    0,
    -1,
    :,
] = -10.0

ignored_positions_logits[
    0,
    -1,
    label2id["B-TOTAL"],
] = 100.0

ignored_positions_results = metrics(
    EvalPrediction(
        predictions=(
            ignored_positions_logits
        ),
        label_ids=metrics_test_label_ids,
    )
)

for metric_name in expected_metric_names:
    assert np.isclose(
        ignored_positions_results[
            metric_name
        ],
        perfect_results[metric_name],
    )

print(
    "Test 5 passed: predictions at -100 "
    "positions do not affect metrics."
)

Test 5 passed: predictions at -100 positions do not affect metrics.


##### Test 6: all-O predictions fail entity extraction

In [167]:
all_o_predictions = np.full(
    metrics_test_label_ids.shape,
    fill_value=label2id["O"],
    dtype=np.int64,
)

all_o_logits = create_test_logits(
    all_o_predictions,
    number_of_labels=number_of_labels,
)

all_o_results = metrics(
    EvalPrediction(
        predictions=all_o_logits,
        label_ids=metrics_test_label_ids,
    )
)

all_o_results

{'precision': np.float64(0.0),
 'recall': np.float64(0.0),
 'f1': np.float64(0.0),
 'token_accuracy': 0.05882352963089943,
 'token_macro_f1': 0.006535947788506746}

Verify the behavior:

In [168]:
assert np.isclose(
    all_o_results["precision"],
    0.0,
)

assert np.isclose(
    all_o_results["recall"],
    0.0,
)

assert np.isclose(
    all_o_results["f1"],
    0.0,
)

expected_token_accuracy = (
    1.0 / number_of_labels
)

assert np.isclose(
    all_o_results["token_accuracy"],
    expected_token_accuracy,
)

assert (
    all_o_results["token_macro_f1"]
    < 1.0
)

print(
    "Test 6 passed: predicting only O "
    "produces zero entity-level metrics."
)

Test 6 passed: predicting only O produces zero entity-level metrics.


This demonstrates why token accuracy alone can be misleading: the model can correctly predict some O tokens while extracting no entities.

#####  Test 7: one incorrect entity token reduces metrics

Change B-INVOICE_NUMBER to O.

In [169]:
partially_wrong_ids = np.where(
    metrics_test_label_ids == -100,
    label2id["O"],
    metrics_test_label_ids,
).copy()

invoice_number_position = np.argwhere(
    metrics_test_label_ids
    == label2id["B-INVOICE_NUMBER"]
)

assert len(invoice_number_position) == 1

batch_index, token_index = (
    invoice_number_position[0]
)

partially_wrong_ids[
    batch_index,
    token_index,
] = label2id["O"]

partially_wrong_logits = create_test_logits(
    partially_wrong_ids,
    number_of_labels=number_of_labels,
)

partially_wrong_results = metrics(
    EvalPrediction(
        predictions=partially_wrong_logits,
        label_ids=metrics_test_label_ids,
    )
)

assert (
    partially_wrong_results["f1"]
    < perfect_results["f1"]
)

assert (
    partially_wrong_results[
        "token_accuracy"
    ]
    < perfect_results["token_accuracy"]
)

assert (
    partially_wrong_results[
        "token_macro_f1"
    ]
    < perfect_results["token_macro_f1"]
)

print(
    "Test 7 passed: an incorrect entity "
    "prediction reduces evaluation metrics."
)

partially_wrong_results

Test 7 passed: an incorrect entity prediction reduces evaluation metrics.


{'precision': np.float64(0.875),
 'recall': np.float64(0.875),
 'f1': np.float64(0.875),
 'token_accuracy': 0.9411764740943909,
 'token_macro_f1': 0.9215686321258545}

#####  Test 8: tuple predictions are supported

Trainer can sometimes pass predictions as a tuple whose first element contains the logits.

In [170]:
tuple_evaluation = EvalPrediction(
    predictions=(
        perfect_logits,
        np.array([123]),
    ),
    label_ids=metrics_test_label_ids,
)

tuple_results = metrics(
    tuple_evaluation
)

assert tuple_results == perfect_results

print(
    "Test 8 passed: tuple predictions "
    "use their first element as logits."
)

Test 8 passed: tuple predictions use their first element as logits.


#####  Test 9: returned values are finite probabilities

In [171]:
for result_name, results in {
    "perfect": perfect_results,
    "all_o": all_o_results,
    "partially_wrong": (
        partially_wrong_results
    ),
}.items():
    for metric_name, metric_value in (
        results.items()
    ):
        assert np.isfinite(metric_value), (
            f"{result_name}/{metric_name} "
            "is not finite."
        )

        assert 0.0 <= metric_value <= 1.0, (
            f"{result_name}/{metric_name} "
            f"is outside [0, 1]: "
            f"{metric_value}"
        )

print(
    "Test 9 passed: all metrics are finite "
    "values between 0 and 1."
)

Test 9 passed: all metrics are finite values between 0 and 1.


##### Test 10: use labels from the real encoder

This verifies that the metrics class accepts the labels generated by `TokenLabelBatchEncoder`.

In [173]:
real_encoded = (
    dataset_batch_encoder({
        "ocred_text": [
            text
        ],
        "json_data": [
            target
        ],
    })
)

real_label_ids = np.asarray(
    real_encoded["labels"],
    dtype=np.int64,
)

real_perfect_logits = create_test_logits(
    real_label_ids,
    number_of_labels=number_of_labels,
    ignored_prediction=label2id["O"],
)

real_results = metrics(
    EvalPrediction(
        predictions=real_perfect_logits,
        label_ids=real_label_ids,
    )
)

assert np.isclose(
    real_results["precision"],
    1.0,
)

assert np.isclose(
    real_results["recall"],
    1.0,
)

assert np.isclose(
    real_results["f1"],
    1.0,
)

assert np.isclose(
    real_results["token_accuracy"],
    1.0,
)

print(
    "Test 10 passed: metrics work with "
    "real encoder labels."
)

real_results

Test 10 passed: metrics work with real encoder labels.


{'precision': np.float64(1.0),
 'recall': np.float64(1.0),
 'f1': np.float64(1.0),
 'token_accuracy': 1.0,
 'token_macro_f1': 1.0}

##### Comparison report

In [174]:
metrics_test_report = pd.DataFrame([
    {
        "scenario": "perfect",
        **perfect_results,
    },
    {
        "scenario": "all O",
        **all_o_results,
    },
    {
        "scenario": "one entity error",
        **partially_wrong_results,
    },
    {
        "scenario": "real encoder labels",
        **real_results,
    },
])


metrics_test_report.style.format({
    "precision": "{:.2%}",
    "recall": "{:.2%}",
    "f1": "{:.2%}",
    "token_accuracy": "{:.2%}",
    "token_macro_f1": "{:.2%}",
})

,scenario,precision,recall,f1,token_accuracy,token_macro_f1
0,perfect,100.00%,100.00%,100.00%,100.00%,100.00%
1,all O,0.00%,0.00%,0.00%,5.88%,0.65%
2,one entity error,87.50%,87.50%,87.50%,94.12%,92.16%
3,real encoder labels,100.00%,100.00%,100.00%,100.00%,100.00%


### Trainer class

In [194]:
from dataclasses import field
from datasets import DatasetDict
from transformers import (
    DataCollatorForTokenClassification,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)

@dataclass(slots=True)
class DocumentModelTrainer:
    model: PreTrainedModel
    tokenizer: PreTrainedTokenizerBase
    datasets: DatasetDict
    training_arguments: TrainingArguments
    metrics: TokenClassificationMetrics

    trainer: Trainer = field(
        init=False,
        repr=False,
    )

    def __post_init__(self) -> None:
        required_splits = {
            "train",
            "validation",
            "test",
        }

        missing_splits = (
            required_splits
            - set(self.datasets)
        )

        if missing_splits:
            raise ValueError(
                "Missing dataset splits: "
                f"{sorted(missing_splits)}"
            )

        data_collator = (
            DataCollatorForTokenClassification(
                tokenizer=self.tokenizer
            )
        )

        self.trainer = Trainer(
            model=self.model,
            args=self.training_arguments,
            train_dataset=(
                self.datasets["train"]
            ),
            eval_dataset=(
                self.datasets["validation"]
            ),
            data_collator=data_collator,
            processing_class=self.tokenizer,
            compute_metrics=self.metrics,
        )

    def train(self):
        return self.trainer.train()

    def evaluate_validation(
        self,
    ) -> dict[str, float]:
        return self.trainer.evaluate(
            eval_dataset=(
                self.datasets["validation"]
            ),
            metric_key_prefix="validation",
        )

    def evaluate_test(
        self,
    ) -> dict[str, float]:
        return self.trainer.evaluate(
            eval_dataset=self.datasets["test"],
            metric_key_prefix="test",
        )

    def predict_dataset(
        self,
        split: str = "test",
    ):
        if split not in self.datasets:
            raise KeyError(
                f"Unknown dataset split: {split}"
            )

        return self.trainer.predict(
            self.datasets[split],
            metric_key_prefix=split,
        )

    def save(
        self,
        output_directory: str,
    ) -> None:
        self.trainer.save_model(
            output_directory
        )

        self.tokenizer.save_pretrained(
            output_directory
        )


#### Test

##### Setup

In [190]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import DatasetDict
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    BertConfig,
    PreTrainedModel,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

In [178]:
row = df.iloc[14]
trainer_test_text = row['ocred_text']
trainer_test_text

'Invoice no: 86164702 Date of issue: 05/15/2020 Seller: Client: Peterson, Hester and Mckee Hernandez, Wright and Pena Unit 6986 Box 0053 01185 Sherry Manor DPO AE 88791 Robertburgh, IL 76443 Tax Id: 910-72-7762 Tax Id: 914-91-3852 IBAN: GB49JBKJ55747171436200 ITEMS No. Description Qty UM Net price Net worth VAT [%] Gross worth Dell Optiplex All in One 5,00 each 129,99 649,95 10% 714,95 Windows 10 Desktop Computer 780 3.OGHz 4GB 250GB 19 LCD SUMMARY VAT [%] Net worth VAT Gross worth 10% 649,95 65,00 714,95 Total $ 649,95 $ 65,00 $ 714,95'

In [179]:
trainer_test_target = InvoiceTarget(
    invoice=InvoiceData(
        invoice_number="86164702",
        invoice_date="05/15/2020",
        seller_name=(
            "Peterson, Hester and Mckee"
        ),
        seller_address=(
            "Unit 6986 Box 0053\n"
            "DPO AE 88791"
        ),
        client_name=(
            "Hernandez, Wright and Pena"
        ),
        client_address=(
            "01185 Sherry Manor\n"
            "Robertburgh, IL 76443"
        ),
    ),
    subtotal=SubtotalData(
        tax="65.00",
        total="714.95",
    ),
    payment_instructions=(
        PaymentInstructions()
    ),
    items=[],
)

In [180]:
trainer_test_fields = {
    str(entity): str(value)
    for entity, value
    in trainer_test_target.extract_fields()
}

expected_trainer_test_fields = {
    "INVOICE_NUMBER": "86164702",
    "INVOICE_DATE": "05/15/2020",
    "SELLER_NAME": (
        "Peterson, Hester and Mckee"
    ),
    "SELLER_ADDRESS": (
        "Unit 6986 Box 0053\n"
        "DPO AE 88791"
    ),
    "CLIENT_NAME": (
        "Hernandez, Wright and Pena"
    ),
    "CLIENT_ADDRESS": (
        "01185 Sherry Manor\n"
        "Robertburgh, IL 76443"
    ),
    "TAX": "65.00",
    "TOTAL": "714.95",
}

assert (
    trainer_test_fields
    == expected_trainer_test_fields
)

print(trainer_test_fields)

{'INVOICE_NUMBER': '86164702', 'INVOICE_DATE': '05/15/2020', 'SELLER_NAME': 'Peterson, Hester and Mckee', 'SELLER_ADDRESS': 'Unit 6986 Box 0053\nDPO AE 88791', 'CLIENT_NAME': 'Hernandez, Wright and Pena', 'CLIENT_ADDRESS': '01185 Sherry Manor\nRobertburgh, IL 76443', 'TAX': '65.00', 'TOTAL': '714.95'}


##### Label configuration

In [181]:
trainer_test_entities = [
    "INVOICE_NUMBER",
    "INVOICE_DATE",
    "SELLER_NAME",
    "SELLER_ADDRESS",
    "CLIENT_NAME",
    "CLIENT_ADDRESS",
    "TAX",
    "TOTAL",
]

trainer_test_labels = ["O"]

for entity in trainer_test_entities:
    trainer_test_labels.extend([
        f"B-{entity}",
        f"I-{entity}",
    ])

trainer_test_label2id = {
    label: index
    for index, label
    in enumerate(trainer_test_labels)
}

trainer_test_id2label = {
    index: label
    for label, index
    in trainer_test_label2id.items()
}

print(trainer_test_labels)

['O', 'B-INVOICE_NUMBER', 'I-INVOICE_NUMBER', 'B-INVOICE_DATE', 'I-INVOICE_DATE', 'B-SELLER_NAME', 'I-SELLER_NAME', 'B-SELLER_ADDRESS', 'I-SELLER_ADDRESS', 'B-CLIENT_NAME', 'I-CLIENT_NAME', 'B-CLIENT_ADDRESS', 'I-CLIENT_ADDRESS', 'B-TAX', 'I-TAX', 'B-TOTAL', 'I-TOTAL']


##### Tokenizer, aligner and encoder

In [182]:
trainer_test_tokenizer = (
    AutoTokenizer.from_pretrained(
        "google-bert/bert-base-uncased",
        use_fast=True,
    )
)

assert trainer_test_tokenizer.is_fast

trainer_test_aligner = DocumentAligner(
    financial_summary_aligner=(
        FinancialSummaryAligner()
    ),
    selector=CandidatesSelector(),
)

trainer_test_encoder = (
    TokenLabelBatchEncoder(
        tokenizer=trainer_test_tokenizer,
        aligner=trainer_test_aligner,
        label2id=trainer_test_label2id,
        max_length=512,
        window_stride=64,
    )
)

Verify alignment before creating datasets:

In [185]:
trainer_test_spans = (
    trainer_test_aligner.find_spans(
        text=trainer_test_text,
        target=trainer_test_target,
    )
)

trainer_test_span_counts = Counter(
    f"B-{span.entity}"
    for span in trainer_test_spans
)

expected_trainer_span_counts = Counter({
    "B-INVOICE_NUMBER": 1,
    "B-INVOICE_DATE": 1,
    "B-SELLER_NAME": 1,
    "B-SELLER_ADDRESS": 2,
    "B-CLIENT_NAME": 1,
    "B-CLIENT_ADDRESS": 2,
    "B-TAX": 1,
    "B-TOTAL": 1,
})

assert (
    trainer_test_span_counts
    == expected_trainer_span_counts
)

pd.DataFrame([
    {
        "entity": str(span.entity),
        "value": span.value,
        "start": span.start,
        "end": span.end,
    }
    for span in trainer_test_spans
])


,entity,value,start,end
0,INVOICE_NUMBER,86164702,12,20
1,INVOICE_DATE,05/15/2020,36,46
2,SELLER_NAME,"Peterson, Hester and Mckee",63,89
3,CLIENT_NAME,"Hernandez, Wright and Pena",90,116
4,SELLER_ADDRESS,Unit 6986 Box 0053,117,135
5,CLIENT_ADDRESS,01185 Sherry Manor,136,154
6,SELLER_ADDRESS,DPO AE 88791,155,167
7,CLIENT_ADDRESS,"Robertburgh, IL 76443",168,189
8,TAX,"65,00",527,532
9,TOTAL,"714,95",535,541


##### Create datasets

Several copies are used so that training has multiple batches. This is appropriate for testing trainer mechanics, but not for evaluating model quality.

In [186]:
trainer_test_record = {
    "ocred_text": trainer_test_text,
    "json_data": (
        trainer_test_target.model_dump_json()
    ),
}

trainer_train_df = pd.DataFrame([
    trainer_test_record,
    trainer_test_record,
    trainer_test_record,
    trainer_test_record,
])

trainer_validation_df = pd.DataFrame([
    trainer_test_record,
])

trainer_test_df = pd.DataFrame([
    trainer_test_record,
])

trainer_dataset_builder = (
    HuggingFaceDatasetBuilder(
        batch_encoder=trainer_test_encoder,
        batch_size=2,
    )
)

trainer_dataset_bundle = (
    trainer_dataset_builder.build(
        train=trainer_train_df,
        validation=trainer_validation_df,
        test=trainer_test_df,
    )
)

trainer_test_datasets = (
    trainer_dataset_bundle.tokenized
)

print(trainer_test_datasets)

/usr/local/lib/python3.13/dist-packages/dill/_dill.py:423: PicklingWarning: Cannot locate reference to <enum 'EntityType'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.13/dist-packages/dill/_dill.py:423: PicklingWarning: Cannot pickle <enum 'EntityType'>: __main__.EntityType has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.13/dist-packages/dill/_dill.py:423: PicklingWarning: Cannot locate reference to <class '__main__.InvoiceData'>.
  StockPickler.save(self, obj, save_persistent_id)
/usr/local/lib/python3.13/dist-packages/dill/_dill.py:423: PicklingWarning: Cannot pickle <class '__main__.InvoiceData'>: __main__.InvoiceData has recursive self-references that trigger a RecursionError.
  StockPickler.save(self, obj, save_persistent_id)


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
})


Verify the splits:

In [187]:
assert set(trainer_test_datasets) == {
    "train",
    "validation",
    "test",
}

assert len(
    trainer_test_datasets["train"]
) == 4

assert len(
    trainer_test_datasets["validation"]
) == 1

assert len(
    trainer_test_datasets["test"]
) == 1


##### Metrics dependency

In [188]:
trainer_test_metrics = (
    TokenClassificationMetrics(
        id2label=trainer_test_id2label,
    )
)


##### Create a small real BERT model

Using full `bert-base-uncased` would make these tests unnecessarily slow. This creates an actual `BertForTokenClassification` through the Hugging Face auto-model API, but with one small transformer layer.

In [191]:
trainer_test_config = BertConfig(
    vocab_size=(
        trainer_test_tokenizer.vocab_size
    ),
    hidden_size=48,
    num_hidden_layers=1,
    num_attention_heads=4,
    intermediate_size=96,
    max_position_embeddings=512,
    hidden_dropout_prob=0.0,
    attention_probs_dropout_prob=0.0,
    pad_token_id=(
        trainer_test_tokenizer.pad_token_id
    ),
    num_labels=len(
        trainer_test_label2id
    ),
    label2id=trainer_test_label2id,
    id2label=trainer_test_id2label,
)

trainer_test_model = (
    AutoModelForTokenClassification
    .from_config(trainer_test_config)
)

assert isinstance(
    trainer_test_model,
    PreTrainedModel,
)

#####  Training arguments

Use a temporary directory so the test does not overwrite a real checkpoint.

In [192]:
trainer_test_root = Path(
    tempfile.mkdtemp(
        prefix="document-model-trainer-"
    )
)

trainer_test_output_directory = (
    trainer_test_root / "training"
)

trainer_test_training_arguments = (
    TrainingArguments(
        output_dir=str(
            trainer_test_output_directory
        ),
        max_steps=2,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=1,
        learning_rate=5e-4,
        logging_strategy="no",
        save_strategy="no",
        eval_strategy="no",
        report_to=[],
        disable_tqdm=True,
        seed=42,
        data_seed=42,
        dataloader_num_workers=0,
    )
)

print(
    "Temporary output:",
    trainer_test_output_directory,
)

Temporary output: /tmp/document-model-trainer-05gtnv_1/training


##### Initialize `DocumentModelTrainer`

In [195]:
document_model_trainer = (
    DocumentModelTrainer(
        model=trainer_test_model,
        tokenizer=trainer_test_tokenizer,
        datasets=trainer_test_datasets,
        training_arguments=(
            trainer_test_training_arguments
        ),
        metrics=trainer_test_metrics,
    )
)

##### Test 1: trainer initialization

In [196]:
assert isinstance(
    document_model_trainer.trainer,
    Trainer,
)

assert (
    document_model_trainer.trainer.model
    is trainer_test_model
)

assert len(
    document_model_trainer
    .trainer
    .train_dataset
) == 4

assert len(
    document_model_trainer
    .trainer
    .eval_dataset
) == 1

assert (
    document_model_trainer
    .trainer
    .compute_metrics
    is trainer_test_metrics
)

print(
    "Test 1 passed: Trainer was initialized "
    "with the expected dependencies."
)

Test 1 passed: Trainer was initialized with the expected dependencies.


#####  Test 2: missing dataset splits are rejected

In [197]:
incomplete_datasets = DatasetDict({
    "train": (
        trainer_test_datasets["train"]
    ),
    "validation": (
        trainer_test_datasets["validation"]
    ),
})

try:
    DocumentModelTrainer(
        model=trainer_test_model,
        tokenizer=trainer_test_tokenizer,
        datasets=incomplete_datasets,
        training_arguments=(
            trainer_test_training_arguments
        ),
        metrics=trainer_test_metrics,
    )
except ValueError as error:
    assert "test" in str(error)
else:
    raise AssertionError(
        "Expected a ValueError for the "
        "missing test split."
    )

print(
    "Test 2 passed: missing dataset splits "
    "are rejected."
)

Test 2 passed: missing dataset splits are rejected.


#####  Test 3: unknown prediction split is rejected

In [198]:
try:
    document_model_trainer.predict_dataset(
        split="unknown"
    )
except KeyError as error:
    assert "unknown" in str(error)
else:
    raise AssertionError(
        "Expected a KeyError for an "
        "unknown dataset split."
    )

print(
    "Test 3 passed: unknown prediction "
    "splits are rejected."
)


Test 3 passed: unknown prediction splits are rejected.


##### Test 4: prediction output is valid

This uses the randomly initialized model, so the quality metrics will initially be poor. The test checks structure, not accuracy.

In [199]:
prediction_output = (
    document_model_trainer.predict_dataset(
        split="test"
    )
)

assert prediction_output.predictions.ndim == 3
assert prediction_output.label_ids.ndim == 2

assert (
    prediction_output.predictions.shape[:2]
    == prediction_output.label_ids.shape
)

assert (
    prediction_output.predictions.shape[-1]
    == len(trainer_test_label2id)
)

required_prediction_metrics = {
    "test_loss",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_token_accuracy",
    "test_token_macro_f1",
}

assert required_prediction_metrics.issubset(
    prediction_output.metrics
)

print(
    "Test 4 passed: predict_dataset() returned "
    "valid logits, labels and metrics."
)

print(prediction_output.metrics)

Test 4 passed: predict_dataset() returned valid logits, labels and metrics.
{'test_loss': 2.778221368789673, 'test_model_preparation_time': 0.001, 'test_precision': 0.0, 'test_recall': 0.0, 'test_f1': 0.0, 'test_token_accuracy': 0.14358974993228912, 'test_token_macro_f1': 0.043309129774570465, 'test_runtime': 1.3855, 'test_samples_per_second': 0.722, 'test_steps_per_second': 0.722}


##### Test 5: validation evaluation

In [200]:
validation_results = (
    document_model_trainer
    .evaluate_validation()
)

required_validation_metrics = {
    "validation_loss",
    "validation_precision",
    "validation_recall",
    "validation_f1",
    "validation_token_accuracy",
    "validation_token_macro_f1",
}

assert required_validation_metrics.issubset(
    validation_results
)

assert np.isfinite(
    validation_results[
        "validation_loss"
    ]
)

for metric_name in [
    "validation_precision",
    "validation_recall",
    "validation_f1",
    "validation_token_accuracy",
    "validation_token_macro_f1",
]:
    assert (
        0.0
        <= validation_results[metric_name]
        <= 1.0
    )

print(
    "Test 5 passed: validation evaluation "
    "returned valid metrics."
)

print(validation_results)

{'validation_loss': '2.778', 'validation_model_preparation_time': '0.001', 'validation_precision': '0', 'validation_recall': '0', 'validation_f1': '0', 'validation_token_accuracy': '0.1436', 'validation_token_macro_f1': '0.04331', 'validation_runtime': '0.0078', 'validation_samples_per_second': '128.6', 'validation_steps_per_second': '128.6', 'epoch': 0}
Test 5 passed: validation evaluation returned valid metrics.
{'validation_loss': 2.778221368789673, 'validation_model_preparation_time': 0.001, 'validation_precision': 0.0, 'validation_recall': 0.0, 'validation_f1': 0.0, 'validation_token_accuracy': 0.14358974993228912, 'validation_token_macro_f1': 0.043309129774570465, 'validation_runtime': 0.0078, 'validation_samples_per_second': 128.565, 'validation_steps_per_second': 128.565, 'epoch': 0}


##### Test 6: test evaluation

In [201]:
test_results = (
    document_model_trainer.evaluate_test()
)

required_test_metrics = {
    "test_loss",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_token_accuracy",
    "test_token_macro_f1",
}

assert required_test_metrics.issubset(
    test_results
)

assert np.isfinite(
    test_results["test_loss"]
)

for metric_name in [
    "test_precision",
    "test_recall",
    "test_f1",
    "test_token_accuracy",
    "test_token_macro_f1",
]:
    assert (
        0.0
        <= test_results[metric_name]
        <= 1.0
    )

print(
    "Test 6 passed: test evaluation "
    "returned valid metrics."
)

print(test_results)

{'test_loss': '2.778', 'test_model_preparation_time': '0.001', 'test_precision': '0', 'test_recall': '0', 'test_f1': '0', 'test_token_accuracy': '0.1436', 'test_token_macro_f1': '0.04331', 'test_runtime': '0.0098', 'test_samples_per_second': '102.4', 'test_steps_per_second': '102.4', 'epoch': 0}
Test 6 passed: test evaluation returned valid metrics.
{'test_loss': 2.778221368789673, 'test_model_preparation_time': 0.001, 'test_precision': 0.0, 'test_recall': 0.0, 'test_f1': 0.0, 'test_token_accuracy': 0.14358974993228912, 'test_token_macro_f1': 0.043309129774570465, 'test_runtime': 0.0098, 'test_samples_per_second': 102.363, 'test_steps_per_second': 102.363, 'epoch': 0}


#####  Test 7: model training runs

In [202]:
parameters_before_training = {
    name: parameter.detach().cpu().clone()
    for name, parameter
    in trainer_test_model.named_parameters()
}

train_output = (
    document_model_trainer.train()
)

assert train_output.global_step == 2
assert np.isfinite(train_output.training_loss)

print(
    "Test 7 passed: training completed "
    f"{train_output.global_step} steps."
)

print(train_output)

{'train_runtime': '0.2865', 'train_samples_per_second': '13.96', 'train_steps_per_second': '6.982', 'train_loss': '2.755', 'epoch': '1'}
Test 7 passed: training completed 2 steps.
TrainOutput(global_step=2, training_loss=2.7554590702056885, metrics={'train_runtime': 0.2865, 'train_samples_per_second': 13.964, 'train_steps_per_second': 6.982, 'train_loss': 2.7554590702056885, 'epoch': 1.0})


##### Test 8: training updates model parameters

In [203]:
parameter_changed = any(
    not torch.equal(
        parameters_before_training[name],
        parameter.detach().cpu(),
    )
    for name, parameter
    in trainer_test_model.named_parameters()
)

assert parameter_changed, (
    "No model parameters changed during training."
)

print(
    "Test 8 passed: training updated "
    "model parameters."
)

Test 8 passed: training updated model parameters.


##### Test 9: evaluation still works after training

In [204]:
post_training_results = (
    document_model_trainer.evaluate_test()
)

assert required_test_metrics.issubset(
    post_training_results
)

assert np.isfinite(
    post_training_results["test_loss"]
)

print(
    "Test 9 passed: evaluation works "
    "after training."
)

print(post_training_results)

{'test_loss': '2.71', 'test_model_preparation_time': '0.001', 'test_precision': '0', 'test_recall': '0', 'test_f1': '0', 'test_token_accuracy': '0.2718', 'test_token_macro_f1': '0.07721', 'test_runtime': '0.0098', 'test_samples_per_second': '102.2', 'test_steps_per_second': '102.2', 'epoch': '1'}
Test 9 passed: evaluation works after training.
{'test_loss': 2.7101023197174072, 'test_model_preparation_time': 0.001, 'test_precision': 0.0, 'test_recall': 0.0, 'test_f1': 0.0, 'test_token_accuracy': 0.2717948853969574, 'test_token_macro_f1': 0.07721259444952011, 'test_runtime': 0.0098, 'test_samples_per_second': 102.225, 'test_steps_per_second': 102.225, 'epoch': 1.0}


Do not assert that all metrics improved after only two steps. This test verifies execution, not convergence.

#####  Test 10: model and tokenizer can be saved


In [205]:
trainer_test_save_directory = (
    trainer_test_root / "saved-model"
)

document_model_trainer.save(
    str(trainer_test_save_directory)
)

assert (
    trainer_test_save_directory
    / "config.json"
).exists()

assert (
    trainer_test_save_directory
    / "tokenizer_config.json"
).exists()

model_file_exists = any(
    (
        trainer_test_save_directory
        / filename
    ).exists()
    for filename in [
        "model.safetensors",
        "pytorch_model.bin",
    ]
)

assert model_file_exists

print(
    "Test 10 passed: model and tokenizer "
    "were saved to:"
)

print(trainer_test_save_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Test 10 passed: model and tokenizer were saved to:
/tmp/document-model-trainer-05gtnv_1/saved-model


##### Evaluation comparison

In [206]:
trainer_evaluation_report = pd.DataFrame([
    {
        "stage": "before training",
        "loss": test_results["test_loss"],
        "precision": test_results[
            "test_precision"
        ],
        "recall": test_results[
            "test_recall"
        ],
        "f1": test_results["test_f1"],
        "token_accuracy": test_results[
            "test_token_accuracy"
        ],
        "token_macro_f1": test_results[
            "test_token_macro_f1"
        ],
    },
    {
        "stage": "after training",
        "loss": post_training_results[
            "test_loss"
        ],
        "precision": post_training_results[
            "test_precision"
        ],
        "recall": post_training_results[
            "test_recall"
        ],
        "f1": post_training_results[
            "test_f1"
        ],
        "token_accuracy": (
            post_training_results[
                "test_token_accuracy"
            ]
        ),
        "token_macro_f1": (
            post_training_results[
                "test_token_macro_f1"
            ]
        ),
    },
])

trainer_evaluation_report

,stage,loss,precision,recall,f1,token_accuracy,token_macro_f1
0,before training,2.778221,0.0,0.0,0.0,0.143590,0.043309
1,after training,2.710102,0.0,0.0,0.0,0.271795,0.077213


### Model predictor

In [207]:
@dataclass(frozen=True, slots=True)
class PredictedEntity:
    entity: EntityType
    start: int
    end: int
    text: str
    confidence: float

    def __post_init__(self) -> None:
        if self.start < 0:
            raise ValueError(
                "Entity start cannot be negative"
            )

        if self.end <= self.start:
            raise ValueError(
                f"Invalid entity span: "
                f"[{self.start}, {self.end})"
            )

        if not 0.0 <= self.confidence <= 1.0:
            raise ValueError(
                "Confidence must be between 0 and 1"
            )

@dataclass(slots=True)
class DocumentModelPredictor:
    model: PreTrainedModel
    tokenizer: PreTrainedTokenizerBase
    id2label: Mapping[int, str]
    max_length: int = 512
    window_stride: int = 64

    def __post_init__(self) -> None:
        if not self.tokenizer.is_fast:
            raise ValueError(
                "DocumentsModelPredictor requires "
                "a fast tokenizer because it uses "
                "offset mappings"
            )

        if self.max_length <= 0:
            raise ValueError(
                "max_length must be positive"
            )

        if not 0 <= self.window_stride < self.max_length:
            raise ValueError(
                "window_stride must be between 0 "
                "and max_length - 1"
            )

    @property
    def device(self) -> torch.device:
        return next(
            self.model.parameters()
        ).device

    def predict(
        self,
        text: str,
    ) -> list[PredictedEntity]:
        text = str(text)

        if not text.strip():
            return []

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            stride=self.window_stride,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding=True,
            return_tensors="pt",
        )

        offset_chunks = encoding.pop(
            "offset_mapping"
        ).tolist()

        encoding.pop(
            "overflow_to_sample_mapping",
            None,
        )

        model_inputs = {
            name: values.to(self.device)
            for name, values in encoding.items()
        }

        self.model.eval()

        with torch.inference_mode():
            logits = self.model(
                **model_inputs
            ).logits.cpu()

        token_predictions = (
            self._merge_chunk_predictions(
                logits=logits,
                offset_chunks=offset_chunks,
            )
        )

        return self._tokens_to_entities(
            text=text,
            token_predictions=(
                token_predictions
            ),
        )

    def _merge_chunk_predictions(
        self,
        *,
        logits: torch.Tensor,
        offset_chunks: list[
            list[list[int]]
        ],
    ) -> list[
        tuple[int, int, str, float]
    ]:
        """
        Merge predictions for tokens repeated in
        overlapping tokenizer chunks.

        Returns:

            [
                (
                    token_start,
                    token_end,
                    predicted_label,
                    confidence,
                ),
                ...
            ]
        """
        logits_by_offset: dict[
            tuple[int, int],
            list[torch.Tensor],
        ] = defaultdict(list)

        for chunk_logits, offsets in zip(
            logits,
            offset_chunks,
        ):
            for token_logits, offset in zip(
                chunk_logits,
                offsets,
            ):
                start, end = offset

                # Ignore special and padding tokens.
                if start == end:
                    continue

                logits_by_offset[
                    (start, end)
                ].append(token_logits)

        predictions = []

        for (
            start,
            end,
        ), repeated_logits in sorted(
            logits_by_offset.items(),
            key=lambda item: item[0],
        ):
            # A token may be present in multiple
            # overlapping chunks. Average its logits.
            mean_logits = torch.stack(
                repeated_logits
            ).mean(dim=0)

            probabilities = torch.softmax(
                mean_logits,
                dim=-1,
            )

            predicted_id = int(
                probabilities.argmax().item()
            )

            confidence = float(
                probabilities[
                    predicted_id
                ].item()
            )

            label = self.id2label[
                predicted_id
            ]

            predictions.append(
                (
                    start,
                    end,
                    label,
                    confidence,
                )
            )

        return predictions

    def _tokens_to_entities(
        self,
        *,
        text: str,
        token_predictions: list[
            tuple[int, int, str, float]
        ],
    ) -> list[PredictedEntity]:
        entities: list[PredictedEntity] = []

        current_entity: EntityType | None = None
        current_start: int | None = None
        current_end: int | None = None
        current_confidences: list[float] = []

        def finish_current_entity() -> None:
            nonlocal current_entity
            nonlocal current_start
            nonlocal current_end
            nonlocal current_confidences

            if (
                current_entity is None
                or current_start is None
                or current_end is None
            ):
                return

            confidence = (
                sum(current_confidences)
                / len(current_confidences)
            )

            entities.append(
                PredictedEntity(
                    entity=current_entity,
                    start=current_start,
                    end=current_end,
                    text=text[
                        current_start:current_end
                    ],
                    confidence=confidence,
                )
            )

            current_entity = None
            current_start = None
            current_end = None
            current_confidences = []

        for (
            token_start,
            token_end,
            label,
            confidence,
        ) in token_predictions:
            if label == "O":
                finish_current_entity()
                continue

            if "-" not in label:
                raise ValueError(
                    f"Invalid BIO label: {label}"
                )

            prefix, entity_name = label.split(
                "-",
                1,
            )

            if prefix not in {"B", "I"}:
                raise ValueError(
                    f"Invalid BIO prefix: {prefix}"
                )

            entity_type = EntityType(
                entity_name
            )

            starts_new_entity = (
                prefix == "B"
                or current_entity is None
                or current_entity != entity_type
            )

            if starts_new_entity:
                finish_current_entity()

                current_entity = entity_type
                current_start = token_start
                current_end = token_end
                current_confidences = [
                    confidence
                ]
            else:
                current_end = token_end
                current_confidences.append(
                    confidence
                )

        finish_current_entity()

        return entities

#### Test

### Output

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from typing import Any

from pydantic import (
    BaseModel,
    ConfigDict,
    Field as PydanticField,
)


class DocumentOutput(BaseModel):
    """
    Structured prediction returned by the document model.

    Fields may be missing because model inference is not
    guaranteed to find every entity.

    `items=None` means line-item extraction is not currently
    supported by this model.
    """

    model_config = ConfigDict(
        extra="forbid",
    )

    invoice: InvoiceData = PydanticField(
        default_factory=InvoiceData
    )

    subtotal: SubtotalData = PydanticField(
        default_factory=SubtotalData
    )

    payment_instructions: (
        PaymentInstructions
    ) = PydanticField(
        default_factory=PaymentInstructions
    )

    items: list[dict[str, Any]] | None = None

    def to_dict(self) -> dict[str, Any]:
        return self.model_dump(
            exclude_none=True
        )

    def to_json(
        self,
        *,
        indent: int | None = 2,
    ) -> str:
        return self.model_dump_json(
            exclude_none=True,
            indent=indent,
        )

@dataclass(frozen=True, slots=True)
class DocumentOutputFormatter:
    def format(
        self,
        entities: list[PredictedEntity],
    ) -> DocumentOutput:
        grouped = self._group_entities(
            entities
        )

        return DocumentOutput(
            invoice=InvoiceData(
                invoice_number=self._first(
                    grouped,
                    EntityType.INVOICE_NUMBER,
                ),
                invoice_date=self._first(
                    grouped,
                    EntityType.INVOICE_DATE,
                ),
                invoice_due_date=self._first(
                    grouped,
                    EntityType.INVOICE_DUE_DATE,
                ),
                seller_name=self._first(
                    grouped,
                    EntityType.SELLER_NAME,
                ),
                seller_address=self._multiline(
                    grouped,
                    EntityType.SELLER_ADDRESS,
                ),
                client_name=self._first(
                    grouped,
                    EntityType.CLIENT_NAME,
                ),
                client_address=self._multiline(
                    grouped,
                    EntityType.CLIENT_ADDRESS,
                ),
            ),
            subtotal=SubtotalData(
                tax=self._last(
                    grouped,
                    EntityType.TAX,
                ),
                total=self._last(
                    grouped,
                    EntityType.TOTAL,
                ),
                discount=self._last(
                    grouped,
                    EntityType.DISCOUNT,
                ),
            ),
            payment_instructions=(
                PaymentInstructions(
                    due_date=self._first(
                        grouped,
                        EntityType.PAYMENT_DUE_DATE,
                    ),
                    account_number=self._first(
                        grouped,
                        EntityType.ACCOUNT_NUMBER,
                    ),
                    bank_name=self._first(
                        grouped,
                        EntityType.BANK_NAME,
                    ),
                    payment_method=self._first(
                        grouped,
                        EntityType.PAYMENT_METHOD,
                    ),
                )
            ),
            items=None,
        )

    def format_json(
        self,
        entities: list[PredictedEntity],
        *,
        indent: int | None = 2,
    ) -> str:
        output = self.format(entities)

        return output.model_dump_json(
            exclude_none=True,
            indent=indent,
        )

    @staticmethod
    def _group_entities(
        entities: list[PredictedEntity],
    ) -> dict[
        EntityType,
        list[PredictedEntity],
    ]:
        grouped: dict[
            EntityType,
            list[PredictedEntity],
        ] = defaultdict(list)

        for entity in sorted(
            entities,
            key=lambda item: item.start,
        ):
            grouped[entity.entity].append(
                entity
            )

        return dict(grouped)

    @staticmethod
    def _first(
        grouped: dict[
            EntityType,
            list[PredictedEntity],
        ],
        entity_type: EntityType,
    ) -> str | None:
        candidates = grouped.get(
            entity_type,
            [],
        )

        if not candidates:
            return None

        return candidates[0].text.strip()

    @staticmethod
    def _last(
        grouped: dict[
            EntityType,
            list[PredictedEntity],
        ],
        entity_type: EntityType,
    ) -> str | None:
        candidates = grouped.get(
            entity_type,
            [],
        )

        if not candidates:
            return None

        return candidates[-1].text.strip()

    @staticmethod
    def _multiline(
        grouped: dict[
            EntityType,
            list[PredictedEntity],
        ],
        entity_type: EntityType,
    ) -> str | None:
        candidates = grouped.get(
            entity_type,
            [],
        )

        values = [
            candidate.text.strip()
            for candidate in candidates
            if candidate.text.strip()
        ]

        if not values:
            return None

        return "\n".join(values)

In [ ]:
import re
import unicodedata

from decimal import Decimal, InvalidOperation


def normalize_money(
    value: object,
) -> str:
    normalized = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    normalized = re.sub(
        r"[^\d,.\-]",
        "",
        normalized,
    )

    if "," in normalized and "." in normalized:
        # The final separator is treated as the
        # decimal separator.
        if (
            normalized.rfind(",")
            > normalized.rfind(".")
        ):
            normalized = (
                normalized
                .replace(".", "")
                .replace(",", ".")
            )
        else:
            normalized = (
                normalized.replace(",", "")
            )

    elif "," in normalized:
        normalized = normalized.replace(
            ",",
            ".",
        )

    try:
        return str(
            Decimal(normalized).normalize()
        )
    except InvalidOperation:
        return normalized

In [ ]:
MONEY_FIELDS = {
    "tax",
    "total",
    "subtotal",
    "discount",
}

IDENTIFIER_FIELDS = {
    "invoice_number",
    "account_number",
}

def normalize_value(
    field: str,
    value: object,
) -> str | None:
    if value is None:
        return None

    normalized = unicodedata.normalize(
        "NFKC",
        str(value),
    ).strip()

    leaf_name = field.split(".")[-1]

    if leaf_name in MONEY_FIELDS:
        return normalize_money(
            normalized
        )

    if leaf_name in IDENTIFIER_FIELDS:
        return re.sub(
            r"[^A-Za-z0-9]",
            "",
            normalized,
        ).upper()

    # Treat newlines and repeated spaces equally.
    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    )

    normalized = normalized.casefold()

    # Normalize common OCR punctuation differences.
    normalized = normalized.replace(
        ":",
        ".",
    )

    normalized = re.sub(
        r"\s*([,.;])\s*",
        r"\1",
        normalized,
    )

    return normalized

In [ ]:
from collections.abc import Mapping
from typing import Any

from pydantic import BaseModel


def document_to_dict(
    document: BaseModel | Mapping[str, Any],
) -> dict[str, Any]:
    if isinstance(document, BaseModel):
        return document.model_dump(
            exclude_none=True
        )

    return dict(document)


def flatten_json(
    document: BaseModel | Mapping[str, Any],
    prefix: str = "",
) -> dict[str, Any]:
    source = document_to_dict(
        document
    )

    result: dict[str, Any] = {}

    for key, value in source.items():
        path = (
            f"{prefix}.{key}"
            if prefix
            else key
        )

        if isinstance(value, BaseModel):
            result.update(
                flatten_json(
                    value,
                    path,
                )
            )

        elif isinstance(value, Mapping):
            result.update(
                flatten_json(
                    value,
                    path,
                )
            )

        elif not isinstance(value, list):
            result[path] = value

    return result

from collections.abc import Iterable, Mapping
from typing import Any

import numpy as np
import pandas as pd
from pydantic import BaseModel


DocumentData = (
    BaseModel
    | Mapping[str, Any]
)


def evaluate_json_records(
    records: Iterable[
        Mapping[str, Any]
    ],
) -> pd.DataFrame:
    records = list(records)

    flattened_records: list[
        tuple[
            dict[str, Any],
            dict[str, Any],
        ]
    ] = []

    field_names: set[str] = set()

    for record in records:
        actual = flatten_json(
            record["actual"]
        )

        predicted = flatten_json(
            record["predicted"]
        )

        flattened_records.append(
            (actual, predicted)
        )

        field_names.update(actual)
        field_names.update(predicted)

    rows = []

    for field in sorted(field_names):
        true_positives = 0
        false_positives = 0
        false_negatives = 0
        expected_count = 0

        for actual, predicted in flattened_records:
            actual_value = actual.get(
                field
            )

            predicted_value = predicted.get(
                field
            )

            actual_present = (
                actual_value not in (None, "")
            )

            predicted_present = (
                predicted_value
                not in (None, "")
            )

            if actual_present:
                expected_count += 1

            correct = (
                actual_present
                and predicted_present
                and normalize_value(
                    field,
                    actual_value,
                )
                == normalize_value(
                    field,
                    predicted_value,
                )
            )

            if correct:
                true_positives += 1
                continue

            if predicted_present:
                false_positives += 1

            if actual_present:
                false_negatives += 1

        precision_denominator = (
            true_positives
            + false_positives
        )

        recall_denominator = (
            true_positives
            + false_negatives
        )

        precision = (
            true_positives
            / precision_denominator
            if precision_denominator
            else 0.0
        )

        recall = (
            true_positives
            / recall_denominator
            if recall_denominator
            else 0.0
        )

        f1 = (
            2
            * precision
            * recall
            / (precision + recall)
            if precision + recall
            else 0.0
        )

        exact_match_rate = (
            true_positives
            / expected_count
            if expected_count
            else np.nan
        )

        rows.append({
            "field": field,
            "support": expected_count,
            "exact_matches": (
                true_positives
            ),
            "exact_match_rate": (
                exact_match_rate
            ),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })

    return pd.DataFrame(rows)

In [ ]:
def document_is_exact(
    record: Mapping[str, Any],
    *,
    penalize_extra_fields: bool = False,
) -> bool:
    actual = flatten_json(
        record["actual"]
    )

    predicted = flatten_json(
        record["predicted"]
    )

    # Every expected field must be present and correct.
    for field, expected_value in actual.items():
        predicted_value = predicted.get(
            field
        )

        if (
            normalize_value(
                field,
                expected_value,
            )
            != normalize_value(
                field,
                predicted_value,
            )
        ):
            return False

    if penalize_extra_fields:
        for field, predicted_value in predicted.items():
            predicted_present = (
                predicted_value
                not in (None, "")
            )

            actual_present = (
                actual.get(field)
                not in (None, "")
            )

            if (
                predicted_present
                and not actual_present
            ):
                return False

    return True

#### Test

### Model evaluation report

In [ ]:
from seqeval.metrics import classification_report

@dataclass(slots=True)
class ModelEvaluationReport:
    summary: dict[str, float]
    entity_metrics: pd.DataFrame
    field_metrics: pd.DataFrame
    document_accuracy: float
    records: pd.DataFrame

@dataclass(slots=True)
class ModelEvaluator:
    trainer: DocumentModelTrainer
    metrics: TokenClassificationMetrics
    predictor: DocumentModelPredictor
    output_formatter: DocumentOutputFormatter

    def evaluate(
        self,
        test_frame: pd.DataFrame,
    ) -> ModelEvaluationReport:
        prediction_output = (
            self.trainer.predict_dataset(
                "test"
            )
        )

        predictions = (
            prediction_output.predictions
        )

        if isinstance(predictions, tuple):
            predictions = predictions[0]

        predicted_labels, actual_labels = (
            self.metrics.decode(
                predictions=predictions,
                label_ids=(
                    prediction_output.label_ids
                ),
            )
        )

        entity_metrics = pd.DataFrame(
            classification_report(
                actual_labels,
                predicted_labels,
                output_dict=True,
                zero_division=0,
            )
        ).transpose()

        evaluation_records = (
            self._build_field_records(
                test_frame
            )
        )

        field_metrics = (
            evaluate_json_records(
                evaluation_records
            )
        )

        document_accuracy = float(
            np.mean([
                document_is_exact(record)
                for record
                in evaluation_records
            ])
        )

        return ModelEvaluationReport(
            summary=dict(
                prediction_output.metrics
            ),
            entity_metrics=entity_metrics,
            field_metrics=field_metrics,
            document_accuracy=(
                document_accuracy
            ),
            records=pd.DataFrame(
                evaluation_records
            ),
        )

    def _build_field_records(
        self,
        frame: pd.DataFrame,
    ) -> list[dict]:
        records = []

        for row_index, row in frame.iterrows():
            target = (
                InvoiceTarget
                .model_validate_json(
                    row["json_data"]
                )
            )

            predicted_entities = (
                self.predictor
                .predict(
                    str(row["ocred_text"])
                )
            )

            predicted_output = (
                self.output_formatter.format(
                    predicted_entities
                )
            )

            records.append({
                "row_index": row_index,
                "actual": target.model_dump(
                    exclude_none=True
                ),
                "predicted": (
                    predicted_output.model_dump(
                        exclude_none=True
                    )
                ),
            })

        return records